<link href="https://fonts.googleapis.com/css2?family=Oswald:wght@700&display=swap" rel="stylesheet">

<h1 style="
font-family: 'Oswald', sans-serif;
font-weight: 700;
font-style: italic;
font-size: 90px;
letter-spacing: 2px;
color: #E7C173;
text-shadow: 3px 3px 0 #333;
">
MACHINE LEARNING<br>IN INDUSTRY
</h1>

# Day 1: Data Preprocessing and Feature Engineering

This notebook is intentionally focused on **data preparation and feature engineering** (no model benchmarking).
The main flow uses `adult_income_issues.csv`, then we transfer the same reasoning to an Ames pipeline capstone.

---

### Table of Contents

- [0. Scope and Success Criteria](#scope)
- [1. Data Intake and Baseline Audit](#audit)
- [2. Splitting Strategy and Leakage Awareness](#splitting)
- [3. Categorical Encoding](#encoding)
- [4. Missing Values](#missing)
- [5. Invalid Values and Outliers](#outliers)
- [6. Scaling and Numeric Transformations](#scaling)
- [7. Feature Engineering](#engineering)
- [8. Feature Selection](#selection)
- [9. End-to-End Pipeline (Ames)](#ames-pipeline)
- [10. Preprocessing Checklist](#checklist)
- [Acceptance Checks](#acceptance)

## <a id="scope"></a> Section 0 - Scope and Success Criteria

**Scope for Day 1**
- Data audit and preprocessing by category.
- Leakage-safe transformation design.
- Feature engineering and qualitative feature selection.
- No model-comparison section (reserved for Day 2).

**Success criteria**
- You can identify preprocessing needs by category (not by manual column-by-column trial and error).
- You can fit transformations on train only and apply them to val/test safely.
- You can explain tradeoffs across encoding, imputation, outlier handling, scaling, and feature design.

In [ ]:
import os
from pathlib import Path

_root = Path.cwd()
while _root != _root.parent and not (_root / "pyproject.toml").exists():
    _root = _root.parent
os.chdir(_root)
print(f"Working directory: {Path.cwd()}")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import (
    OrdinalEncoder,
    OneHotEncoder,
    StandardScaler,
    MinMaxScaler,
    RobustScaler,
    PowerTransformer,
    KBinsDiscretizer,
    PolynomialFeatures,
)
from sklearn.impute import SimpleImputer, KNNImputer, MissingIndicator
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif
from sklearn.decomposition import PCA
from sklearn.feature_extraction import FeatureHasher

# IterativeImputer is experimental in sklearn.
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

SEED = 42
np.random.seed(SEED)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)

try:
    import category_encoders as ce
    HAS_CATEGORY_ENCODERS = True
except Exception:
    HAS_CATEGORY_ENCODERS = False


def show(df, n=5):
    display(df.head(n))


def to_numeric_loose(series: pd.Series) -> pd.Series:
    cleaned = series.astype("string").str.strip().str.replace("h", "", regex=False)
    return pd.to_numeric(cleaned, errors="coerce")


print("Environment ready. category_encoders installed:", HAS_CATEGORY_ENCODERS)


## <a id="audit"></a> Section 1 - Data Intake and Baseline Audit (Adult main)


<div style="max-height:300px; overflow:auto; border:1px solid #ddd; padding:6px">

| Dataset | Task | Issues file (rows x cols) | Main issues |
|---|---|---:|---|
| Adult Income | Binary classification (<span style="color:#c0392b;font-weight:bold">`class`</span>: `<=50K` / `>50K`) | `adult_income_issues.csv` (50,317 x 36) | Missingness, leakage, duplicates, mixed types, invalid values, rare/new categories, sparse indicators, process artifacts |
| Ames Housing | Regression (<span style="color:#c0392b;font-weight:bold">`sale_price`</span>) | `ames_housing_issues.csv` (1,576 x 84) | Truncation/capping, multiple rows per entity |
| Retail Panel | Time-aware store-level forecasting (<span style="color:#c0392b;font-weight:bold">`sales`</span>) | `retail_panel_issues.csv` (22,800 x 9) | Data drift, time misalignment, process artifacts |

</div>

### Adult Income (`adult_income_issues.csv`)
Person-level application/income dataset with added operational and quality issues. Target is binary: <span style="color:#c0392b;font-weight:bold">`class`</span> is either `<=50K` or `>50k`. 

-----

The first things one usually checks are:

- which columns are present in the data
- what's the data type of each column
- how many unique values does each column have
- how many missing values does each column have
- whether the raw dtype hides a better semantic type (for example numeric/date/boolean values stored as strings)

In [ ]:
adult = pd.read_csv("day1/generated/adult_income_issues.csv")
ames = pd.read_csv("day1/generated/ames_housing_issues.csv")
retail = pd.read_csv("day1/generated/retail_panel_issues.csv")

TARGET_COL = "class"
TARGET_BIN_COL = "target"
SPLIT_COL = "split"
ID_COLS = ["person_id"]

LEAKAGE_COLS = [
    "post_adjudication_risk_code",  
]

PROCESS_COLS = [
    "db_source_table",
    "db_etl_batch_id",
    "db_row_surrogate_key",
    "db_loaded_at_utc",
    "dataset_schema_version",
    "extract_country_code",
    "record_written_at",
    "dgp_regime",
]

adult[TARGET_BIN_COL] = adult[TARGET_COL].astype(str).str.contains(">50", case=False, regex=False).astype(int)

print("Adult shape:", adult.shape)
print("Ames shape:", ames.shape)
print("Retail shape:", retail.shape)
print("Adult target positive rate:", round(adult[TARGET_BIN_COL].mean(), 4))


In [ ]:
def build_basic_audit_table(df: pd.DataFrame) -> pd.DataFrame:

    """
    Build a basic audit table for a DataFrame, summarizing key characteristics of each column, including:
    Column name, raw data type, number and percentage of missing values, number and percentage of unique values,
    percentage of values parseable as numeric, datetime, and boolean, average string length (for non-missing values), and example values.
    """

    bool_tokens = {"true", "false", "yes", "no", "y", "n", "t", "f", "0", "1"}
    dateish_pattern = r"[-/:]|\\b(?:jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)\\b"

    rows = []
    for col in df.columns:
        s = df[col]
        raw = s.astype("string").str.strip()
        non_missing = raw.dropna()

        numeric_ratio = pd.to_numeric(non_missing, errors="coerce").notna().mean() if len(non_missing) else 0.0
        dateish_mask = non_missing.str.contains(dateish_pattern, case=False, regex=True, na=False)
        datetime_ratio = pd.to_datetime(non_missing.where(dateish_mask), errors="coerce", format="mixed").notna().mean() if len(non_missing) else 0.0
        boolean_ratio = non_missing.str.lower().isin(bool_tokens).mean() if len(non_missing) else 0.0
        avg_len = non_missing.str.len().mean() if len(non_missing) else 0.0

        rows.append({
            "column": col,
            "raw_dtype": str(s.dtype),
            "n_missing": int(s.isna().sum()),
            "pct_missing": round(100 * float(s.isna().mean()), 2),
            "n_unique": int(s.nunique(dropna=True)),
            "n_unique_including_na": int(raw.fillna("__MISSING__").nunique()),
            "pct_unique": round(100 * float(s.nunique(dropna=True) / len(s)), 2),
            "pct_parseable_numeric": round(100 * float(numeric_ratio), 2),
            "pct_parseable_datetime": round(100 * float(datetime_ratio), 2),
            "pct_parseable_boolean": round(100 * float(boolean_ratio), 2),
            "avg_string_length": round(float(avg_len), 1) if len(non_missing) else 0.0,
            "example_values": non_missing.head(3).tolist(),
        })

    return pd.DataFrame(rows)

audit_adult = build_basic_audit_table(adult).sort_values("column")
show(audit_adult, 3)

In [ ]:
# Here we show how some column types might not be correct and we identify the proper one. 
# This is pretty crucial for modeling: the way we treat a categorical field vs a numeric one is very different.

columns = ['column', 'raw_dtype', 'n_unique', 'pct_unique', 'pct_parseable_numeric']

audit_adult["string_but_numeric"] = audit_adult["raw_dtype"].isin(["str", "object", "string"]) & (audit_adult["pct_parseable_numeric"] >= 90.0)
audit_adult["string_but_datetime"] = audit_adult["raw_dtype"].isin(["str", "object", "string"]) & (audit_adult["pct_parseable_datetime"] >= 90.0)
audit_adult["string_but_boolean"] = audit_adult["raw_dtype"].isin(["str", "object", "string"]) & (audit_adult["pct_parseable_boolean"] >= 95.0) & (audit_adult["n_unique"] <= 5)

coercion_candidates = audit_adult.loc[
    audit_adult["string_but_numeric"] | audit_adult["string_but_datetime"] | audit_adult["string_but_boolean"],
    [
        "column",
        "raw_dtype",
        "string_but_numeric",
        "string_but_datetime",
        "string_but_boolean",
        "example_values",
    ],
]
show(coercion_candidates, n=20)


##### Make the first decisions after the first look at the data.

- We need to identify the **target** column (you know this beforehand)

- **Cast columns** whose raw data type is not the proper one when inspecting their values (e.g. str columns made of just numbers).

- We need to identify a candidate **ID/entity column**, if present: you do not want to use that as a feature, because the model would overfit the training data, and new IDs at prediction time would be meaningless.

- We should **drop exact duplicate rows**: they add no information and can distort frequencies.

- If the same **ID/entity value** appears multiple times, we should decide what one row means. Here we first inspect which columns change within the same ID; since `record_written_at` behaves like a snapshot timestamp, we keep the latest row for each repeated ID.

- We can **drop constant/always missing columns**: if this is all the data that the model sees during training and validation, even if a new value appears at testing/production time, the model will not know what to do with it.

- We can **drop columns that are not known at prediction time**. Let's say that the model uses `record_written_at` as a feature, but in the production dgp it is not present, or maybe it has a different format: we cannot use it.

- We can **drop columns that are leaking the target** (check Bonus Feature later): in real world datasets tables are usually not built for a data scientist to build models on. To make an example, in datasets about loans, there might be a column `defaulted` which could be the target for a Loan Default model. But together with that we might have columns that are populated just when the loan defaulted: `default_date`, `collection_started_at` (trying to recover the money from the borrower), `days_past_due_at_default`.

Usually, before dropping columns we should ask ourselves:

1. Is the column constant?
2. Is the column available on the data used in production?
3. Is the column leaking the target?

For some columns we might not know whether to keep them or not. Columns like `record_written_at`, `db_loaded_batch_id`, or `dgp_regime`, even if they do not represent real business context, might still be hiding some latent factor and help the model, so we can keep them under review.



###### Cast Columns

In [ ]:
bool_map = {"true": True, "false": False, "yes": True, "no": False, "y": True, "n": False, "t": True, "f": False, "1": True, "0": False}

for c in audit_adult.loc[audit_adult["string_but_numeric"], "column"]:
    adult[c] = to_numeric_loose(adult[c])

for c in audit_adult.loc[audit_adult["string_but_datetime"], "column"]:
    adult[c] = pd.to_datetime(adult[c], errors="coerce", format="mixed")

for c in audit_adult.loc[audit_adult["string_but_boolean"], "column"]:
    adult[c] = adult[c].astype("string").str.strip().str.lower().map(bool_map).astype("boolean")


###### Drop Exact Duplicates

No need to have the same id and information twice, that will inflate any kind of model. Beware that theoretically we might encounter 2 different instances/IDs with the same exact feature set (highly unlikely the more data you use). 

In [ ]:
target_cols = [c for c in [TARGET_COL, TARGET_BIN_COL] if c in adult.columns]
id_cols = [c for c in ID_COLS if c in adult.columns]

n_exact_duplicate_rows = int(adult.duplicated().sum())
adult_raw = adult.copy()
adult = adult.drop_duplicates(ignore_index=True).copy()
print("Dropped exact duplicate rows:", n_exact_duplicate_rows)


###### Handle duplicate IDs

If the same ID is present twice with 2 different records, how do we pick which one to keep?

In [ ]:
adult_step = adult.copy()

duplicate_id_change_summary = pd.DataFrame(columns=["column", "avg_n_unique_per_person"])

if id_cols:
    id_col = id_cols[0]
    duplicated_ids = adult_step.loc[adult_step.duplicated(subset=[id_col], keep=False)]
    non_id_cols = [c for c in adult.columns if c not in id_cols]
    duplicated_ids[non_id_cols] = duplicated_ids[non_id_cols].astype(str)
    if len(duplicated_ids) > 0:
        duplicate_id_change_columns = (
            duplicated_ids
            .fillna("missing") # nunique does not consider null values 
            .groupby(id_col).nunique()
            #.mean(axis=0)
            #.rename("avg_n_unique_per_person")
            .reset_index().rename(columns={"index": "column"}))
        
        counts_per_id = (duplicated_ids[id_col]
                         .value_counts().rename('n_rows_per_id')
                         .reset_index()
                         .rename(columns={"index": "column"}))
        
        duplicate_id_change_summary = (
            duplicate_id_change_columns
            .merge(counts_per_id, on=id_col, how="left")

            #.sort_values("avg_n_unique_per_person", ascending=False)
        )
        cols_ = duplicate_id_change_summary.drop([id_col, 'n_rows_per_id'], axis=1).columns.tolist()
        duplicate_id_change_summary = (duplicate_id_change_summary[cols_].div(duplicate_id_change_summary["n_rows_per_id"], axis=0)
        .mean(axis=0).rename('average_ratio_distinct_values_per_id').reset_index()
        .rename(columns={"index": "column"}))

show(duplicate_id_change_summary.sort_values('average_ratio_distinct_values_per_id', ascending=False), n=5)


We can see that record_written_at is always different given the same person_id. Idea could be to keep the latest, so that we don't have the same person_id twice. 

**Note:** it might be that there is no clear logic on how to separate the records. As long as your choices are documented and acknowledged that's good.

In [ ]:
adult = (adult.sort_values("record_written_at", ascending=False).drop_duplicates("person_id", keep="first").reset_index(drop=True))

###### Drop constant and leakage columns

In [ ]:
print("Adult shape after row-level cleaning:", adult.shape)

audit_adult_clean = build_basic_audit_table(adult).sort_values("column")
show(audit_adult_clean.sort_values('pct_unique', ascending=False), n=3)
constant_cols = audit_adult_clean.loc[
    audit_adult_clean["n_unique_including_na"] <= 1,
    "column",
].tolist()

not_available_at_prediction_cols = [
    c for c in [
        "db_row_surrogate_key",
        "db_source_table",
        "db_loaded_at_utc",
        "dataset_schema_version",
        "extract_country_code",
    ]
    if c in adult.columns
]

leakage_cols = [c for c in LEAKAGE_COLS if c in adult.columns]

review_cols = [
    c for c in [
        "record_written_at",
        "db_etl_batch_id",
        "dgp_regime",
    ]
    if c in adult.columns
]

decision_rows = []

for c in target_cols:
    decision_rows.append({"column": c, "decision": "target", "why": "Known beforehand; not a feature"})

for c in id_cols:
    decision_rows.append({"column": c, "decision": "drop", "why": "Identifier column"})

for c in constant_cols:
    decision_rows.append({"column": c, "decision": "drop", "why": "Constant column"})

for c in not_available_at_prediction_cols:
    decision_rows.append({"column": c, "decision": "drop", "why": "Not available at prediction time"})

for c in leakage_cols:
    decision_rows.append({"column": c, "decision": "drop", "why": "Target leakage"})

for c in review_cols:
    decision_rows.append({"column": c, "decision": "review", "why": "Could hide a latent factor; check deployment story"})

first_decisions = (
    pd.DataFrame(decision_rows)
    .drop_duplicates(subset=["column"], keep="first")
    .sort_values(["decision", "column"])
)

show(first_decisions, n=40)

drop_cols = first_decisions.loc[first_decisions["decision"] == "drop", "column"].tolist()

excluded_from_modeling = set(target_cols) | set(drop_cols)
model_candidate_cols = [c for c in adult.columns if c not in excluded_from_modeling]

print("Model candidate columns:", len(model_candidate_cols))
print("Columns under review:", [c for c in review_cols if c in model_candidate_cols])

> **Do It Yourself**
>
> On `retail`, build the same profile and classify columns into: `numeric`, `categorical`, `text`, `date`, `process/metadata`, `target/id`.
> Also flag at least one column where raw dtype and semantic type differ.
>
> Hint: start from `date`, then compare your answer with the worked Adult example below.

######

In [ ]:
show(retail)

##### **Bonus Feature**

>
> You might not know which columns are leaking the target, because you don't know the domain of the data.
>
> In that case you might want to try one of the following techniques:
> 
> 1. Train a simple model using only that feature. If it gives abnormally high AUC/logloss improvement, it is likely leakage.
> 
> 2. Mutual information / PPS Compute MI or PPS between feature and target. Extremely high dependence for a single feature is suspicious.

###### 1. Train a simple Model

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score

def leakage_tree_score(X, y, feature, cv=5):
    model = DecisionTreeClassifier(max_depth=3)
    return cross_val_score(model, X[[feature]], y, cv=cv, scoring="roc_auc").mean()

for col in adult.drop([TARGET_COL, TARGET_BIN_COL], axis=1).select_dtypes('number').columns:
    single_col = adult[[col, TARGET_BIN_COL]]
    single_col[col] = single_col[col].astype(float)
    ls = (leakage_tree_score(single_col, single_col[TARGET_BIN_COL], col, cv=3))
    if ls > 0.6:
        print(f"Potential leakage detected: {col} with score {ls:.4f}")

###### Mutual information

> [Mutual_information](https://en.wikipedia.org/wiki/Mutual_information)
> 
> Rule of Thumb:
> 1. MI ~ `0 - 0.005` -> noise
> 2. `0.005 - 0.02` -> weak signal
> 3. `0.02 - 0.1`-> useful feature
> 4. `> 0.1` -> very strong predictor
> 5. `> 0.3` -> extremely strong -> investigate leakage

> In practice:
> 
> 1. if one feature is 5-10x larger than the rest, inspect if for leakage
>
> 2. if one feature has MI ~ Target Entropy -> almost certainly leakage

In [ ]:
from sklearn.feature_selection import mutual_info_classif

def feature_mi(X, y):
    X2 = X.copy()
    for c in X2.columns:
        cdtype = X2[c].dtype
        
        if cdtype in ["str", "datetime", "datetime64[us]", "date"]:
            X2[c] = X2[c].astype("category").cat.codes
    X2 = X2.fillna(-999)
    mi = mutual_info_classif(X2, y)
    return (pd.DataFrame(dict(zip(X2.columns, mi)).items(),
                        columns=["feature", "mutual_info"])
                        .assign(mutual_info=lambda df: df["mutual_info"].round(4))  
                        .sort_values("mutual_info", ascending=False))

def target_entropy(y):
    p = np.bincount(y) / len(y)
    p = p[p > 0]
    return -(p * np.log2(p)).sum()

show(feature_mi(adult.drop([TARGET_COL, TARGET_BIN_COL], axis=1), adult[TARGET_BIN_COL]), 10)
print("Target entropy:", target_entropy(adult[TARGET_BIN_COL].values.round(3)))

##### Handling Row-Wise column misalignment

If you remember, in our dataset we had misalignment between our columns, resulting in nonsensical values for some of the columns (like 48 in `relationship` or 'Never-married` in workclass). 

Ideally, we would like to shift the values 'under' the proper column, yet here we'll just put them to null.

In [ ]:
misaligned_cols = ["workclass", "marital_status", "occupation", "relationship"]

allowed_values = {"workclass": ['Private', 'Self-emp-not-inc', 'State-gov', 'Local-gov', 'Federal-gov'],
                  "marital_status": ['Separated','Widowed','Never-married','Married-civ-spouse','Divorced'],
                  "relationship": ['Unmarried', 'Other-relative', 'Husband', 'Wife', 'Own-child', 'Not-in-family'],
                  "occupation": ['Other-service', 'Craft-repair', 'Exec-managerial', 'Sales', 'Adm-clerical', 'Prof-specialty', 'quantum_technician', 'policy_simulator', 'robotics_annotator', 'Respiratory therapist', 'Geospatial surveyor', 'Aircraft painter', 'Court reporter', 'Marine electronics technician', 'Orthotic fitter', 'Food safety inspector']}

for column, values in allowed_values.items():
    adult.loc[~adult[column].astype(str).isin(values), column] = pd.NA

## <a id="splitting"></a> Section 2 - Splitting Strategy and Leakage Awareness

Before doing any preprocessing, we split the data into **train / val / test**.

The key rule: **all transformers and encoders are fitted on the training set only**, then applied to val and test.
If you fit on the full dataset before splitting, statistics from val/test leak into your transforms and your offline metrics become optimistically biased.

**Note:** We'll explore splitting strategies in depth in Day 2 (random vs stratified vs group-aware vs time-based).
For now, we use the dataset's `split` column and derive a validation set from train.

In [ ]:
if SPLIT_COL in adult.columns:
    train_pool = adult.loc[adult[SPLIT_COL] == "train"].copy()
    test_df = adult.loc[adult[SPLIT_COL] == "test"].copy()
else:
    train_pool, test_df = train_test_split(
        adult,
        test_size=0.2,
        random_state=SEED,
        stratify=adult[TARGET_BIN_COL],
    )

# Extra safety check: after upstream cleaning there should be no overlapping IDs across splits.
# If any remain, remove them from train_pool.
overlap_ids = set(train_pool[ID_COLS[0]]) & set(test_df[ID_COLS[0]])
if overlap_ids:
    print(f"Warning: Found {len(overlap_ids)} overlapping IDs across train/test splits. Removing from train pool to ensure split integrity.")
    train_pool = train_pool.loc[~train_pool[ID_COLS[0]].isin(overlap_ids)].copy()
    print(f"Removed {len(overlap_ids)} overlapping IDs from train pool to enforce split integrity.")

# stratified split between train and validation

id_series = train_pool[ID_COLS[0]].dropna()
id_target = train_pool.groupby(ID_COLS[0])[TARGET_BIN_COL].mean().round().astype(int)
strat = id_target.values if id_target.nunique() > 1 else None

id_train, id_val = train_test_split(
    id_target.index.to_numpy(),
    test_size=0.2,
    random_state=SEED,
    stratify=strat,
)

train_df = train_pool.loc[train_pool[ID_COLS[0]].isin(id_train)].copy()
val_df = train_pool.loc[train_pool[ID_COLS[0]].isin(id_val)].copy()

for name, frame in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name:>5} rows={len(frame):5d} target_rate={frame[TARGET_BIN_COL].mean():.4f}")

train_ids = set(train_df[ID_COLS[0]])
val_ids = set(val_df[ID_COLS[0]])
test_ids = set(test_df[ID_COLS[0]])

assert len(train_ids & val_ids) == 0
assert len(train_ids & test_ids) == 0
assert len(val_ids & test_ids) == 0

print("Split integrity check passed (no ID overlap across train/val/test).")



In [ ]:
split_summary = pd.DataFrame({
    "split": ["train", "val", "test"],
    "n_rows": [len(train_df), len(val_df), len(test_df)],
    "target_rate": [
        train_df[TARGET_BIN_COL].mean(),
        val_df[TARGET_BIN_COL].mean(),
        test_df[TARGET_BIN_COL].mean(),
    ],
})

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

sns.barplot(data=split_summary, x="split", y="n_rows", ax=axes[0])#, palette="Set2")
axes[0].set_title("Rows per split")
axes[0].set_ylabel("Number of rows")

sns.barplot(data=split_summary, x="split", y="target_rate", ax=axes[1])#, palette="Set2")
axes[1].set_title("Positive label ratio per split")
axes[1].set_ylabel(TARGET_BIN_COL)

for ax, col in zip(axes, ["n_rows", "target_rate"]):
    for i, v in enumerate(split_summary[col]):
        label = f"{v:.3f}" if col == "target_rate" else f"{int(v)}"
        ax.text(i, v, label, ha="center", va="bottom")

plt.tight_layout()
plt.show()


## <a id="encoding"></a> Section 3 - Categorical Encoding (Popular Methods)

We compare label, ordinal, one-hot, frequency, and target encoding.
All mappings/encoders are learned on **train only**.

📚 [Preprocessing Guide](https://scikit-learn.org/stable/modules/preprocessing.html#encoding-categorical-features) · [OrdinalEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OrdinalEncoder.html) · [OneHotEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) · [category_encoders](https://contrib.scikit-learn.org/category_encoders/) · [TargetEncoder](https://contrib.scikit-learn.org/category_encoders/targetencoder.html)

In [ ]:
excluded_cols = set([TARGET_COL, TARGET_BIN_COL, SPLIT_COL] + ID_COLS + LEAKAGE_COLS + PROCESS_COLS)
feature_cols = [c for c in adult.columns if c not in excluded_cols]

numeric_like_cols = []
categorical_cols = []
for col in feature_cols:
    s = train_df[col]
    if s.dtype.kind in "biufc": # biufc means boolean, integer, unsigned integer, float, complex
        numeric_like_cols.append(col)
        continue
    as_num = to_numeric_loose(s)
    if as_num.notna().mean() >= 0.9:
        numeric_like_cols.append(col)
    else:
        categorical_cols.append(col)

categorical_cols = [c for c in categorical_cols if c not in ["case_review_note"]]

X_train_raw = train_df[feature_cols].copy()
X_val_raw = val_df[feature_cols].copy()
X_test_raw = test_df[feature_cols].copy()

y_train = train_df[TARGET_BIN_COL].copy()
y_val = val_df[TARGET_BIN_COL].copy()
y_test = test_df[TARGET_BIN_COL].copy()

print("Feature columns:", len(feature_cols))
print("Numeric-like columns:", len(numeric_like_cols))
print("Categorical columns:", len(categorical_cols))
print(f"Categorical columns: {', '.join(categorical_cols)}")

show(pd.DataFrame({
    "original_dtype": train_df[feature_cols].dtypes.astype(str),
    "processed_as": ["numeric_like" if c in numeric_like_cols else "categorical" for c in feature_cols],
    "processed_dtype": [str(to_numeric_loose(train_df[c]).dtype) if c in numeric_like_cols else str(train_df[c].dtype) for c in feature_cols],
}, index=feature_cols), 20)

# Numeric-ready base data for standalone technique cells (Sections 3-7)
X_train_num = pd.DataFrame({c: to_numeric_loose(X_train_raw[c]) for c in numeric_like_cols}, index=X_train_raw.index)
X_val_num = pd.DataFrame({c: to_numeric_loose(X_val_raw[c]) for c in numeric_like_cols}, index=X_val_raw.index)
X_test_num = pd.DataFrame({c: to_numeric_loose(X_test_raw[c]) for c in numeric_like_cols}, index=X_test_raw.index)

print("Numeric base shapes:", X_train_num.shape, X_val_num.shape, X_test_num.shape)

In [ ]:
#little helper to check if we have unseen categories in test vs train

def unseen_categories(train_series: pd.Series, other_series: pd.Series) -> list:
    a = set(train_series.dropna().astype(str).unique())
    b = set(other_series.dropna().astype(str).unique())
    return sorted(b - a)

cat_cols_for_check = [c for c in model_candidate_cols if c in adult.columns and adult[c].dtype == "object"]

if "occupation" in adult.columns:
    unseen_occ_test = unseen_categories(train_df["occupation"], test_df["occupation"])
    print("Unseen 'occupation' categories in test vs train:", unseen_occ_test)
    assert len(unseen_occ_test) > 0, "Expected at least one unseen category in test."

> **Do It Yourself**
>
> Check the same on all categorical columns in the dataset.
> 1. Check if split proportions and target prevalence are reasonable.
> 2. Find at least one categorical feature where test has unseen labels compared to train.


######

In [ ]:
#cat_cols = list(set(adult.select_dtypes(include=["object", "str", "string"]).columns.tolist())&(set(model_candidate_cols)))
#for cat_col in cat_cols:
#    unseen = unseen_categories(train_df[cat_col], test_df[cat_col])
#    print(f"Unseen categories in '{cat_col}' (test vs train): {unseen}")

##### Ordinal Encoding

[scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OrdinalEncoder.html)

*What it does*: It transforms a categorical variable with $N$ distinct values into integers from $0$ to $N-1$.

> Example 1.:
>
> `regione`: ['Lombardia', 'Lazio', 'Puglia', 'Sicilia'] 
>
> $N=4$
>
> `regione_encoded`: [0, 1, 2, 3]

> Example 2.:
>
> `education`: ['elementary', 'middle school', 'high school', 'bachelor', 'master', 'doctorate'] 
>
> $N=6$
>
> `education_encoded`: [0, 1, 2, 3, 4, 5]

*When to use it*: when categories have a real order and we are able to estimate the distance between each category, or with tree-based models. 

*What to pay attention to*: for nominal categories, the numbers are arbitrary and may create fake order/distances. Can we say that Puglia is "twice" Lazio or that Lombardia is "less" than Sicilia? Is the distance between middle school and elementary the same as the one between doctorate and master?

*Model implications*: linear models and neural networks can be misled; trees are usually less affected. 

In the following example we create a new categorical column from `hours_per_week`, named `hours_band`. 


In [ ]:
def build_hours_band(series: pd.Series) -> pd.Series:
    hours = to_numeric_loose(series)
    return pd.cut(
        hours,
        bins=[-np.inf, 30, 45, 60, np.inf],
        labels=["very_low", "low", "medium", "high"],
        ordered=True,
    )

# Work on local copies — X_train_raw stays unchanged
_train = X_train_raw.copy()
_val = X_val_raw.copy()
_test = X_test_raw.copy()

for frame in [_train, _val, _test]:
    frame["hours_band"] = build_hours_band(frame["hours_per_week"])

ord_col = "hours_band"
ord_categories = [["very_low", "low", "medium", "high"]]
ord_enc = OrdinalEncoder(categories=ord_categories, handle_unknown="use_encoded_value", unknown_value=-1)

X_train_ord = ord_enc.fit_transform(_train[[ord_col]])
X_val_ord = ord_enc.transform(_val[[ord_col]])
X_test_ord = ord_enc.transform(_test[[ord_col]])

show(pd.concat([pd.DataFrame(X_train_ord, columns=['hours_band_encoded']).reset_index(drop=True),
                 _train[[ord_col, 'hours_per_week']].reset_index(drop=True)], axis=1)
                 .drop_duplicates('hours_band')
                 .sort_values('hours_per_week', ignore_index=True), 6)

##### One-Hot Encoding

[scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html)

*What it does*: It transforms a categorical variable with $N$ distinct values into $N$ binary columns, one for each category.

> Example:
>
> `regione`: ['Lombardia', 'Lazio', 'Puglia', 'Sicilia']
>
> $N=4$
>
> `regione_encoded`: `regione_Lombardia`, `regione_Lazio`, `regione_Puglia`, `regione_Sicilia`
>
> `Puglia` becomes `[0, 0, 1, 0]`

*When to use it*: when categories are nominal and have low or medium cardinality, especially with linear models, neural networks, and distance-based models.

*What to pay attention to*: it can create many columns, produce sparse matrices, and become expensive with high-cardinality features. Unseen categories need explicit handling, for example with `handle_unknown="ignore"`.

*Handling rare categories*: sklearn's `OneHotEncoder` has two built-in parameters for this:
- `min_frequency` — categories appearing fewer than this many times (or this fraction of rows) are lumped into a single `infrequent_if_exist` column.
- `max_categories` — keeps at most this many categories (plus the infrequent bucket). Useful when you want a hard cap on dimensionality.

Both require `handle_unknown="infrequent_if_exist"` so that unseen categories at transform time also fall into the infrequent bucket.

*Model implications*: it is usually a strong default for linear models and neural networks. Tree-based models can use it too, but very wide one-hot matrices are often inefficient and may fragment splits.

In [ ]:
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

train_nom = X_train_raw[categorical_cols].fillna("MISSING").astype(str)
val_nom = X_val_raw[categorical_cols].fillna("MISSING").astype(str)
test_nom = X_test_raw[categorical_cols].fillna("MISSING").astype(str)

X_train_ohe = ohe.fit_transform(train_nom)
X_val_ohe = ohe.transform(val_nom)
X_test_ohe = ohe.transform(test_nom)

ohe_cols = ohe.get_feature_names_out(categorical_cols)

X_train_ohe_df = pd.DataFrame(X_train_ohe, index=X_train_raw.index, columns=ohe_cols)
X_val_ohe_df = pd.DataFrame(X_val_ohe, index=X_val_raw.index, columns=ohe_cols)
X_test_ohe_df = pd.DataFrame(X_test_ohe, index=X_test_raw.index, columns=ohe_cols)

print("OHE shapes:", X_train_ohe_df.shape, X_val_ohe_df.shape, X_test_ohe_df.shape)
show(X_train_ohe_df, n=3)

##### Frequency Encoding

*What it does*: It replaces each category with how often that category appears in the training set, either as a count or as a relative frequency.

> Example:
>
> `occupation`: ['admin', 'teacher', 'admin', 'doctor', 'admin', 'teacher']
>
> `occupation_freq`: {'admin': 0.50, 'teacher': 0.33, 'doctor': 0.17}
>
> `teacher` becomes `0.33`

*When to use it*: when categories are nominal and high-cardinality, and you want a compact numeric feature instead of many one-hot columns.

*What to pay attention to*: different categories can receive the same encoded value if they appear equally often. It does not use the target, and it can change if the category distribution shifts or unseen labels appear at inference time.

*Model implications*: it keeps the feature space small, which is convenient for most models. Tree-based models often use it more naturally, while linear models and neural networks may interpret a more frequent category as a larger numeric value even when that has no real meaning.


In [ ]:
# Frequency Encoding
freq_maps = {}
X_train_freq = pd.DataFrame(index=X_train_raw.index)
X_val_freq = pd.DataFrame(index=X_val_raw.index)
X_test_freq = pd.DataFrame(index=X_test_raw.index)

freq_cols = [c for c in ["occupation", "native_country"] if c in X_train_raw.columns]
for col in freq_cols:
    freq = X_train_raw[col].astype(str).value_counts(normalize=True)
    freq_maps[col] = freq
    X_train_freq[f"{col}_freq"] = X_train_raw[col].astype(str).map(freq).fillna(0.0)
    X_val_freq[f"{col}_freq"] = X_val_raw[col].astype(str).map(freq).fillna(0.0)
    X_test_freq[f"{col}_freq"] = X_test_raw[col].astype(str).map(freq).fillna(0.0)

show(pd.concat([X_train_freq, X_train_raw[['occupation', 'native_country']]], axis=1)
     .drop_duplicates(['occupation', 'native_country'], ignore_index=True), n=15)


> **Do It Yourself**
>
> Check on the `ames` dataset whether applying frequency encoding to a categorical variable might mislead the model.
> Hint:
> 1. Grab the categorical columns from the dataset (object, str, dates)
> 2. See the frequency of each level in each categorical variable
> 3. Check the target mean/median/standard deviation for each level
> 
> See if you find 2 different levels (ex. neighborhoods) encoded to a similar value, but with a different target distribution (mean, median, stddev).


######

In [ ]:
target_col_freq = 'sale_price'
#show(ames.groupby('ms_zoning')
#      .agg(nrows=(target_col_freq, 'size'), mean_price=(target_col_freq, 'median'))
#      .reset_index()
#      .assign(frequency_ms_zoning=lambda df: round(df['nrows'] / df['nrows'].sum(), 3))
#      .query('frequency_ms_zoning > 0.002')
#      .sort_values('frequency_ms_zoning'), 20
#      )

##### Target Encoding

[category_encoders](https://contrib.scikit-learn.org/category_encoders/targetencoder.html)

*What it does*: It replaces each category with a statistic of the target for that category, typically the mean target value computed on the training set.

> Example:
>
> `occupation`: ['teacher', 'doctor', 'teacher', 'unemployed']
>
> `target`: [0, 0, 1, 1]
>
> `occupation_te`: {'teacher': 0.50, 'doctor': 0.00, 'unemployed': 1.00}

*When to use it*: when categories are nominal and high-cardinality, one-hot encoding would create too many columns, and the category is expected to carry signal about the target.

*What to pay attention to*: leakage is the main risk. The encoding must be learned on train only, and for the training fold itself it should be computed out-of-fold. Rare categories usually need smoothing toward the global mean.

*Model implications*: it can be very predictive while keeping a single numeric column, so it often works well with tree-based models and can also help linear models. If done naively, however, it can overfit badly and make validation results look much better than they really are.

**Note:** target encoding can be used just if you have a target (duh), so it's not possible to use it in unsupervised problems.

See this nice [Kaggle Notebook](https://www.kaggle.com/code/ryanholbrook/target-encoding).


In [ ]:
def oof_target_encode(train_cat: pd.Series, y: pd.Series, n_splits: int = 5, seed: int = 42):
    train_cat = train_cat.astype(str)
    y = y.astype(float)
    global_mean = y.mean()

    oof = pd.Series(index=train_cat.index, dtype=float)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)

    for tr_idx, ho_idx in kf.split(train_cat):
        tr_c = train_cat.iloc[tr_idx]
        tr_y = y.iloc[tr_idx]
        fold_map = tr_y.groupby(tr_c).mean()
        oof.iloc[ho_idx] = train_cat.iloc[ho_idx].map(fold_map).fillna(global_mean)

    full_map = y.groupby(train_cat).mean()
    return oof, full_map, global_mean


target_maps = {}
if "native_country" in X_train_raw.columns:
    te_col = "native_country"

    # Wrong/leaky: uses full dataset (including val/test information)
    leaky_map = adult.groupby(adult[te_col].astype(str))[TARGET_BIN_COL].mean()
    train_te_leaky = X_train_raw[te_col].astype(str).map(leaky_map).fillna(adult[TARGET_BIN_COL].mean())

    # Safer: train-only + OOF
    train_te_oof, safe_map, safe_global = oof_target_encode(X_train_raw[te_col], y_train, n_splits=5, seed=SEED)
    val_te_safe = X_val_raw[te_col].astype(str).map(safe_map).fillna(safe_global)
    test_te_safe = X_test_raw[te_col].astype(str).map(safe_map).fillna(safe_global)

    target_maps[te_col] = safe_map

    compare = pd.DataFrame({
        "y_train": y_train,
        "te_leaky": train_te_leaky,
        "te_safe_oof": train_te_oof,
    })
    print("Train correlation with target (leaky TE):", round(compare[["y_train", "te_leaky"]].corr().iloc[0, 1], 4))
    print("Train correlation with target (safe OOF TE):", round(compare[["y_train", "te_safe_oof"]].corr().iloc[0, 1], 4))

    show(compare, n=5)


> **Do It Yourself**
>
> Pick one of the datasets and apply target encoding (using the whole training, vs using the OOF method) and see how they correlate with the target.
>
###### 

## <a id="missing"></a> Section 4 - Missing Values (Full Ladder)

Methods covered from simple to advanced:
- native missing-value handling (model-dependent)
- drop rows/columns
- constant fill
- median/mode
- missing indicators
- group-wise imputation
- KNN imputation
- Iterative (MICE-like) imputation

📚 [Imputation Guide](https://scikit-learn.org/stable/modules/impute.html) · [SimpleImputer](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html) · [KNNImputer](https://scikit-learn.org/stable/modules/generated/sklearn.impute.KNNImputer.html) · [IterativeImputer](https://scikit-learn.org/stable/modules/generated/sklearn.impute.IterativeImputer.html) · Rubin (1976), *Inference and Missing Data*

##### Diagnose Missingness First

*What it does*: It measures how much missing data there is and checks whether missingness may be related to other variables.

*When to use it*: always before deciding whether to drop, fill, or leave missing values untouched.

*What to pay attention to*: MCAR, MAR, and MNAR are assumptions about the data-generating process, not labels you can prove with one summary table.

*Model implications*: if missingness itself carries signal, adding indicators or using models that can exploit NaNs can help.


In [ ]:
missing_report = (
    train_df.isna().mean().rename("missing_pct")
    .reset_index()
    .rename(columns={"index": "column"})
    .assign(missing_pct=lambda df: 100*df["missing_pct"])
    .sort_values("missing_pct", ascending=False)
)
show(missing_report, n=15)

# Example cues for missingness mechanisms
if "sale_price" in ames.columns and "reported_max_loan" in ames.columns:
    tmp = ames.assign(sale_price_missing=ames["sale_price"].isna())
    print("Ames loan stats by sale_price missingness:")
    display(tmp.groupby("sale_price_missing")["reported_max_loan"].describe())

print("Interpretation reminder: MCAR/MAR/MNAR is a data-generating assumption, not a test you prove with certainty.")


##### Native Missing-Value Handling

[LightGBM](https://lightgbm.readthedocs.io/en/latest/Advanced-Topics.html#missing-value-handle) · [XGBoost](https://xgboost.readthedocs.io/en/stable/faq.html#how-to-deal-with-missing-values) · [CatBoost](https://catboost.ai/docs/en/concepts/algorithm-missing-values-processing) · [HistGradientBoosting](https://scikit-learn.org/stable/modules/ensemble.html#histogram-based-gradient-boosting)

*What it does*: Some models can learn how to route `NaN` values directly, without replacing them first.

*When to use it*: when you train models with native missing-value support, such as LightGBM, XGBoost tree boosters, or scikit-learn HistGradientBoosting.

*What to pay attention to*: this is model-specific. Many linear models, neural networks, and classic scikit-learn estimators still require explicit handling.

*Model implications*: native-missing tree models may benefit from keeping `NaN` as `NaN`, while a shared pre-imputed matrix is still convenient if you want to compare many different model families.


##### Drop Rows / Columns

*What it does*: It removes observations or features that contain missing values.

*When to use it*: when missingness is very rare, or when a column is so incomplete that keeping it is not worth the complexity.

*What to pay attention to*: it can throw away a lot of information and introduce bias if missingness is not random. If you are asked to provide a prediction for all rows dropping them is a no-go.

*Model implications*: it is simple and model-agnostic, but often too aggressive unless the amount of missing data is small.


In [ ]:
missing_demo_cols = [c for c in ["age", "hours_per_week", "capital_gain", "occupation"] if c in train_df.columns]
missing_demo = train_df[missing_demo_cols].copy()

num_cols_demo = [c for c in missing_demo.columns if to_numeric_loose(missing_demo[c]).notna().mean() >= 0.9]
cat_cols_demo = [c for c in missing_demo.columns if c not in num_cols_demo]

rows_after_drop = len(missing_demo.dropna())
cols_after_drop = missing_demo.dropna(axis=1).shape[1]

print("Rows after dropna:", rows_after_drop)
print("Columns after dropna(axis=1):", cols_after_drop)


##### Constant Fill

*What it does*: It replaces missing values with a fixed placeholder, such as `"MISSING"`, `0`, or `-999`.

*When to use it*: when missingness should become an explicit category, or when you need a quick deterministic baseline.

*What to pay attention to*: the placeholder must not be confused with a real value, and for numeric features a bad sentinel can distort the distribution.

*Model implications*: tree-based models usually tolerate sentinel values better than linear models, but adding a missingness indicator is often safer.


In [ ]:
missing_demo_cols = [c for c in ["age", "hours_per_week", "capital_gain", "occupation"] if c in train_df.columns]
missing_demo = train_df[missing_demo_cols].copy()
num_cols_demo = [c for c in missing_demo.columns if to_numeric_loose(missing_demo[c]).notna().mean() >= 0.9]
cat_cols_demo = [c for c in missing_demo.columns if c not in num_cols_demo]

const_fill = missing_demo.copy()
for c in num_cols_demo:
    const_fill[c] = to_numeric_loose(const_fill[c]).fillna(-999)
for c in cat_cols_demo:
    const_fill[c] = const_fill[c].fillna("MISSING")

show(const_fill.loc[(const_fill.isin(['MISSING', -999])).any(axis=1)], n=5)

##### Median / Mode Imputation

*What it does*: It fills numeric variables with the median and categorical variables with the most frequent category.

*When to use it*: as a strong baseline when missingness is limited and you want something simple, robust, and easy to explain.

*What to pay attention to*: medians reduce variance and modes can hide minority categories. The imputer must be fit on train only. 

*Model implications*: this is a common default for linear models and many scikit-learn models that cannot ingest `NaN` values.


In [ ]:
missing_demo_cols = [c for c in ["age", "hours_per_week", "capital_gain", "occupation"] if c in train_df.columns]
missing_demo = train_df[missing_demo_cols].copy()
num_cols_demo = [c for c in missing_demo.columns if to_numeric_loose(missing_demo[c]).notna().mean() >= 0.9]
cat_cols_demo = [c for c in missing_demo.columns if c not in num_cols_demo]

median_mode = missing_demo.copy()
for c in num_cols_demo:
    median_mode[c] = to_numeric_loose(median_mode[c])
num_imp = SimpleImputer(strategy="median")
cat_imp = SimpleImputer(strategy="most_frequent")

if num_cols_demo:
    median_mode[num_cols_demo] = num_imp.fit_transform(median_mode[num_cols_demo])
if cat_cols_demo:
    median_mode[cat_cols_demo] = cat_imp.fit_transform(median_mode[cat_cols_demo])

show(median_mode, n=5)

In [ ]:
ax = X_train_raw.capital_gain.plot(kind="hist", bins=50, title="capital_gain distribution", figsize=(6, 4))

mean_ = X_train_raw["capital_gain"].mean()
mode_ = X_train_raw["capital_gain"].mode().iloc[0]
median_ = X_train_raw["capital_gain"].median()

ax.axvline(mean_, color="crimson", linestyle="--", label=f"mean = {mean_:.1f}")
ax.axvline(mode_, color="darkgreen", linestyle="--", label=f"mode = {mode_:.1f}")
ax.axvline(median_, color="orange", linestyle="--", label=f"median = {median_:.1f}")
ax.legend();


##### Missing Indicators

*What it does*: It creates a binary feature that marks whether the original value was missing.

*When to use it*: when the absence of a value may itself contain information, especially together with a simple imputation rule.

*What to pay attention to*: adding indicators for too many almost-complete columns can create noise and unnecessary dimensionality.

*Model implications*: indicators often help linear models and neural networks recover signal that would otherwise be hidden by the fill value. Native-missing tree models do not need them (it could help them converge faster to an optimum).

**Note:** `scikit-learn` [MissingIndicator](https://scikit-learn.org/stable/modules/generated/sklearn.impute.MissingIndicator.html) requires input features to be numeric (so non-numeric cols should have already been handled).


In [ ]:
missing_demo_cols = [c for c in ["age", "hours_per_week", "capital_gain", "occupation"] if c in train_df.columns]
missing_demo = train_df[missing_demo_cols].copy()
num_cols_demo = [c for c in missing_demo.columns if to_numeric_loose(missing_demo[c]).notna().mean() >= 0.9]
cat_cols_demo = [c for c in missing_demo.columns if c not in num_cols_demo]

indicator = MissingIndicator(features="all")
num_cols_demo = missing_demo.select_dtypes(include=["number"]).columns.tolist()
missing_demo_num = missing_demo[num_cols_demo].copy()
ind_train = indicator.fit_transform(missing_demo_num.select_dtypes(include=["number"]))

ind_cols = [f"is_missing_{c}" for c in missing_demo_num.columns]
ind_df = pd.DataFrame(ind_train, columns=ind_cols, index=missing_demo_num.index)

idx_ = ind_df.loc[ind_df.any(axis=1)].index

show(pd.concat([missing_demo, ind_df], axis=1).loc[idx_], n=5)

##### Group-Wise Imputation

*What it does*: It fills missing values using statistics computed inside a relevant subgroup, such as median income within the same job category.

*When to use it*: when missing values are likely to depend on another feature and a single global fill would be too crude.

*What to pay attention to*: weak or noisy grouping variables can inject bias, and unseen groups still need a fallback such as the global median.

*Model implications*: it often preserves structure better than one global imputer, while staying simpler and more interpretable than fully model-based imputers.

Read more [here](https://www.kdnuggets.com/2017/09/python-data-preparation-case-files-group-based-imputation.html)


In [ ]:
print(X_train_raw.groupby('workclass').capital_gain.agg(['mean', 'count']))

capital_gain_fill = (X_train_raw[["capital_gain", "workclass"]]
                     .groupby("workclass")['capital_gain'].mean()
                     .rename('group_mean').reset_index())

example_group_fill = X_train_raw[["capital_gain", "workclass"]].copy()
example_group_fill = example_group_fill.merge(capital_gain_fill, on="workclass", how='left')
example_group_fill['capital_gain_filled'] = example_group_fill['capital_gain'].copy()
example_group_fill.loc[example_group_fill.capital_gain.isna(), 'capital_gain_filled'] = example_group_fill['group_mean']

print("\nRows with missing capital_gain by workclass:")
print(example_group_fill.query("capital_gain.isna()").workclass.value_counts())

##### KNN Imputation

*What it does*: It fills a missing value using nearby observations in feature space.

*When to use it*: when numeric observations with similar profiles should have similar missing values, and the dataset is not too large.

*What to pay attention to*: it is sensitive to feature scaling, can be slow on larger datasets, and may behave poorly if neighborhoods are weak or noisy. You are filling one column each time.

*Model implications*: it can preserve local structure better than median imputation, but it adds computational cost and extra preprocessing decisions.


In [ ]:
impute_cols = [c for c in ["age", "hours_per_week", "capital_gain", "capital_loss", "education_num"] if c in train_df.columns]

train_num_imp = pd.DataFrame({c: to_numeric_loose(train_df[c]) for c in impute_cols}, index=train_df.index)
val_num_imp = pd.DataFrame({c: to_numeric_loose(val_df[c]) for c in impute_cols}, index=val_df.index)
test_num_imp = pd.DataFrame({c: to_numeric_loose(test_df[c]) for c in impute_cols}, index=test_df.index)

knn_imp = KNNImputer(n_neighbors=5)
train_knn = pd.DataFrame(knn_imp.fit_transform(train_num_imp), columns=impute_cols, index=train_num_imp.index)
val_knn = pd.DataFrame(knn_imp.transform(val_num_imp), columns=impute_cols, index=val_num_imp.index)
test_knn = pd.DataFrame(knn_imp.transform(test_num_imp), columns=impute_cols, index=test_num_imp.index)

print("KNN missing left in train:", int(train_knn.isna().sum().sum()))
show(train_knn, n=5)


###### Iterative Imputation

*What it does*: It models each variable with missing values as a function of the other variables, then imputes repeatedly.

*When to use it*: when numeric features are correlated and you want a richer, model-based imputation strategy.

*What to pay attention to*: it is slower, less transparent, and more sensitive to modeling assumptions than simple imputers. It still has to be fit on train only.

*Model implications*: it can work well when feature relationships are strong, but the added complexity does not always translate into better downstream performance.


In [ ]:
impute_cols = [c for c in ["age", "hours_per_week", "capital_gain", "capital_loss", "education_num"] if c in train_df.columns]

train_num_imp = pd.DataFrame({c: to_numeric_loose(train_df[c]) for c in impute_cols}, index=train_df.index)
val_num_imp = pd.DataFrame({c: to_numeric_loose(val_df[c]) for c in impute_cols}, index=val_df.index)
test_num_imp = pd.DataFrame({c: to_numeric_loose(test_df[c]) for c in impute_cols}, index=test_df.index)

iter_imp = IterativeImputer(random_state=SEED, max_iter=10)
train_iter = pd.DataFrame(iter_imp.fit_transform(train_num_imp), columns=impute_cols, index=train_num_imp.index)
val_iter = pd.DataFrame(iter_imp.transform(val_num_imp), columns=impute_cols, index=val_num_imp.index)
test_iter = pd.DataFrame(iter_imp.transform(test_num_imp), columns=impute_cols, index=test_num_imp.index)

print("Iterative missing left in train:", int(train_iter.isna().sum().sum()))
show(train_iter, n=5)


> **Do It Yourself**
>
> Define an imputation policy for at least 5 columns:
> - method
> - rationale
> - risk/tradeoff
>
> Hint: include at least one column where you would add a missingness indicator, or justify leaving `NaN` untouched for a model with native missing-value support.

######

## <a id="outliers"></a> Section 5 - Invalid Values and Outliers

We combine:
- rule-based cleaning,
- statistical clipping,
- model-based outlier detection (IsolationForest).

Fix impossible values first. Then decide whether valid extremes should be clipped, flagged, reviewed, or left untouched.

Outliers tend to affect linear models and neural networks more than tree-based models (which care about the ranking of instances, not much about the magnitude). For linear models check [here](https://www.savemyexams.com/ap/statistics/college-board/20/revision-notes/exploring-two-variable-data/scatterplots-and-regression/outliers-high-leverage-and-influential-points/) the difference between an 'influential' and a 'non-influential' outlier.

📚 [IsolationForest](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.IsolationForest.html) · Liu et al. (2008), *Isolation Forest, ICDM*

##### Rule-Based Validation

*What it does*: It applies domain rules to detect impossible or suspicious values and either flags them or converts them to missing values.

*When to use it*: always first, whenever you know valid ranges or business rules for a feature.

*What to pay attention to*: do not confuse rare-but-valid values with true data errors. If the correction itself is informative, keep a flag.

*Model implications*: this is model-agnostic and usually more defensible than blind clipping because it is grounded in domain meaning.


In [ ]:
def apply_rule_cleaning(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()

    if "age" in out.columns:
        age = to_numeric_loose(out["age"])
        bad_age = ((age < 0) | (age > 100)).fillna(False)
        out["age_invalid_flag"] = bad_age.astype(int)
        out["age"] = age.mask(bad_age)

    if "hours_per_week" in out.columns:
        hpw = to_numeric_loose(out["hours_per_week"])
        bad_hpw = ((hpw < 1) | (hpw > 120)).fillna(False)
        out["hours_invalid_flag"] = bad_hpw.astype(int)
        out["hours_per_week"] = hpw.mask(bad_hpw)

    if "capital_gain" in out.columns:
        cg = to_numeric_loose(out["capital_gain"])
        bad_cg = (cg < 0).fillna(False)
        out["capital_gain_invalid_flag"] = bad_cg.astype(int)
        out["capital_gain"] = cg.mask(bad_cg)

    return out

train_rules = apply_rule_cleaning(train_df)
val_rules = apply_rule_cleaning(val_df)
test_rules = apply_rule_cleaning(test_df)

rule_counts = {
    "train_bad_age": int(train_rules.get("age_invalid_flag", pd.Series(dtype=int)).sum()),
    "train_bad_hours": int(train_rules.get("hours_invalid_flag", pd.Series(dtype=int)).sum()),
    "train_bad_capital_gain": int(train_rules.get("capital_gain_invalid_flag", pd.Series(dtype=int)).sum()),
}
print(rule_counts)


##### Statistical Clipping / Capping

Also called [Winsorization](https://www.blog.trainindata.com/winsorization-handling-outliers-in-machine-learning/). 

*What it does*: It caps extreme values using train-fitted thresholds, such as IQR fences, to reduce their leverage on downstream models.

*When to use it*: when values are plausible but so extreme that they distort summaries, scaling, or linear-model fits.

*What to pay attention to*: clipping changes the data and can hide rare but important behavior. Thresholds must be learned on train only.

*Model implications*: it often helps linear models and standard scalers more than tree-based models. The robust z-score counts below are a diagnostic, not a replacement for judgment.


In [ ]:
# Statistical clipping with train-fitted IQR bounds
_train = X_train_num.copy()
_val = X_val_num.copy()
_test = X_test_num.copy()

clip_log = []
clip_cols = _train.columns.tolist()

for col in clip_cols:
    q1 = _train[col].quantile(0.25)
    q3 = _train[col].quantile(0.75)
    iqr = q3 - q1
    if pd.isna(iqr):
        continue
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr

    before_train = int(((_train[col] < low) | (_train[col] > high)).sum())

    _train[col] = _train[col].clip(low, high)
    _val[col] = _val[col].clip(low, high)
    _test[col] = _test[col].clip(low, high)

    clip_log.append({"column": col, "low": low, "high": high, "n_train_clipped": before_train})

clip_log = pd.DataFrame(clip_log).sort_values("n_train_clipped", ascending=False)
show(clip_log, n=10)

# Robust z-score counts (MAD-based)
robust_counts = []
for col in clip_cols[:20]:
    med = _train[col].median()
    mad = (_train[col] - med).abs().median()
    if mad == 0 or pd.isna(mad):
        continue
    rz = 0.6745 * (_train[col] - med) / mad
    robust_counts.append({"column": col, "n_abs_rz_gt_3.5": int((rz.abs() > 3.5).sum())})

robust_counts = pd.DataFrame(robust_counts).sort_values("n_abs_rz_gt_3.5", ascending=False)
show(robust_counts, n=10)

##### Model-Based Outlier Detection

*What it does*: It uses a multivariate model to flag rows that look unusual given the joint feature pattern.

*When to use it*: when single-feature rules are not enough and you want a review signal for unusual combinations of otherwise valid values.

*What to pay attention to*: unsupervised detectors can flag minority but legitimate behavior. The contamination rate is a modeling assumption, not a discovered truth.

*Model implications*: model-based flags are often safer as extra features or review triggers than as automatic row-deletion rules.

[IsolationForest](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.IsolationForest.html).

In [ ]:
# Model-based detector example: IsolationForest
_train = X_train_num.copy()
_val = X_val_num.copy()
_test = X_test_num.copy()

iso_cols = _train.columns.tolist()[:30]

iso = IsolationForest(
    n_estimators=200,
    contamination=0.02,
    random_state=SEED,
)
iso.fit(_train[iso_cols])

train_iso_flag = (iso.predict(_train[iso_cols]) == -1).astype(int)
val_iso_flag = (iso.predict(_val[iso_cols]) == -1).astype(int)
test_iso_flag = (iso.predict(_test[iso_cols]) == -1).astype(int)

print("IsolationForest outlier share (train/val/test):",
      round(train_iso_flag.mean(), 4),
      round(val_iso_flag.mean(), 4),
      round(test_iso_flag.mean(), 4))

> **Do It Yourself**
>
> Create a per-feature policy table with one of:
> - drop row
> - clip/cap
> - set to missing + impute
> - keep + add flag
>
> Hint: prioritize rules for impossible values before statistical outlier methods.

##### Practical Outlier Policy

*What it does*: It turns the previous ideas into a per-feature action such as keep, clip, flag, or set to missing and impute later.

*When to use it*: before building a production pipeline or before handing the data to downstream modeling steps.

*What to pay attention to*: policies should follow domain meaning, expected inference-time behavior, and model sensitivity, not only summary statistics.

*Model implications*: different models tolerate extremes differently, so the best policy can change depending on the modeling family you plan to use.


In [ ]:
outlier_policy = pd.DataFrame([
    {"column": "age", "policy": "set impossible to NaN -> median impute", "reason": "Physical domain limits"},
    {"column": "hours_per_week", "policy": "clip + keep invalid flag", "reason": "Potential data-entry errors"},
    {"column": "capital_gain", "policy": "right-tail clip at IQR upper", "reason": "Extreme skew"},
    {"column": "occupation_te_safe", "policy": "keep as is", "reason": "Already smoothed encoded value"},
])
show(outlier_policy, n=10)

print("Outlier/invalid value policy defined.")

## <a id="scaling"></a> Section 6 - Scaling and Numeric Transformations

We compare Standard, MinMax, and Robust scaling, then add log/power transforms for skewed features.

📚 [Preprocessing Guide](https://scikit-learn.org/stable/modules/preprocessing.html#standardization-or-mean-removal-and-variance-scaling) · [Compare the effect of different scalers on data with outliers](https://scikit-learn.org/stable/auto_examples/preprocessing/plot_all_scaling.html) · [Map data to a normal distribution](https://scikit-learn.org/stable/auto_examples/preprocessing/plot_map_data_to_normal.html) · [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) · [MinMaxScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html) · [RobustScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.RobustScaler.html) · [PowerTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PowerTransformer.html) · Yeo & Johnson (2000), *A New Family of Power Transformations*

##### When Scaling Matters

*What it does*: Scaling changes the numeric units of the features so they become more comparable.

*When to use it*: when the model depends on distances, dot products, regularization strength, gradient descent, or PCA-like decompositions.

*What to pay attention to*: fit the scaler on train only. Tree-based models are usually much less sensitive to scaling than linear, kernel, or distance-based models.

*Model implications*: scaling often matters for linear models with regularization, kNN, SVMs, PCA, and neural networks. It is usually much less important for tree ensembles.


##### Standard Scaling

*What it does*: It subtracts the mean and divides by the standard deviation, so the feature is centered around 0 with unit variance.

*When to use it*: when features are roughly symmetric and you train models that expect comparable scales.

*What to pay attention to*: it is sensitive to outliers, so a few extreme values can distort the mean and standard deviation.

*Model implications*: it is a common default for linear models, SVMs, PCA, and neural networks. Tree-based models usually do not require it.


In [ ]:
_train = X_train_num.copy()
_val = X_val_num.copy()

scale_cols = _train.columns.tolist()
compare_col = scale_cols[0]

std_scaler = StandardScaler()
train_std = pd.DataFrame(std_scaler.fit_transform(_train[scale_cols]), columns=scale_cols, index=_train.index)
val_std = pd.DataFrame(std_scaler.transform(_val[scale_cols]), columns=scale_cols, index=_val.index)

standard_compare = pd.DataFrame({
    "original": _train[compare_col].describe(),
    "standard": train_std[compare_col].describe(),
})
show(standard_compare, n=10)

##### MinMax Scaling

*What it does*: It rescales each feature into a fixed interval, usually `[0, 1]`.

*When to use it*: when bounded inputs are useful, or when you want to preserve rank/order while compressing the numeric range.

*What to pay attention to*: it is very sensitive to outliers, because one extreme value can squeeze most observations into a narrow band. In practice, one huge max or min can collapse the rest of the feature close to 0 or 1 and reduce separation for linear, kernel, or distance-based models.

*Model implications*: it can be useful for some neural-network setups and distance-based models, but it is often fragile when heavy tails are present.

📚 [Scaler comparison with outliers](https://scikit-learn.org/stable/auto_examples/preprocessing/plot_all_scaling.html)

In [ ]:
_train = X_train_num.copy()

scale_cols = _train.columns.tolist()
compare_col = scale_cols[0]

mm_scaler = MinMaxScaler()
train_mm = pd.DataFrame(mm_scaler.fit_transform(_train[scale_cols]), columns=scale_cols, index=_train.index)

minmax_compare = pd.DataFrame({
    "original": _train[compare_col].describe(),
    "minmax": train_mm[compare_col].describe(),
})
show(minmax_compare, n=10)

##### Robust Scaling

*What it does*: It centers features with the median and scales them with the interquartile range.

*When to use it*: when outliers are frequent and you still need scaled inputs for models that are sensitive to feature magnitude.

*What to pay attention to*: it reduces outlier influence, but it does not remove skewness or make a distribution normal. Think of it as a safer scaler under heavy tails, not as an outlier-cleaning method.

*Model implications*: it is often a safer default than standard or min-max scaling on heavy-tailed data. Tree-based models still usually do not need it.

📚 [Scaler comparison with outliers](https://scikit-learn.org/stable/auto_examples/preprocessing/plot_all_scaling.html)

In [ ]:
_train = X_train_num.copy()
_val = X_val_num.copy()
_test = X_test_num.copy()

scale_cols = _train.columns.tolist()
compare_col = scale_cols[0]

rb_scaler = RobustScaler()
train_rb = pd.DataFrame(rb_scaler.fit_transform(_train[scale_cols]), columns=scale_cols, index=_train.index)
val_rb = pd.DataFrame(rb_scaler.transform(_val[scale_cols]), columns=scale_cols, index=_val.index)
test_rb = pd.DataFrame(rb_scaler.transform(_test[scale_cols]), columns=scale_cols, index=_test.index)

robust_compare = pd.DataFrame({
    "original": _train[compare_col].describe(),
    "robust": train_rb[compare_col].describe(),
})
show(robust_compare, n=10)

print("Robust scaling applied.")

##### Log Transform

*What it does*: It compresses large positive values with a `log1p` transform, which often reduces right skew.

*When to use it*: for non-negative variables such as counts, amounts, or monetary features with a long right tail.

*What to pay attention to*: negative values need special handling, and the transformed feature becomes less directly interpretable. A `log1p` transform often helps when a few very large values dominate the scale, but it will not magically fix every distribution.

*Model implications*: log transforms often make linear models and standard scalers behave better on skewed features. Tree models are usually less dependent on them.

📚 [Power and quantile transforms example](https://scikit-learn.org/stable/auto_examples/preprocessing/plot_map_data_to_normal.html)

In [ ]:
_train = X_train_num.copy()
_val = X_val_num.copy()
_test = X_test_num.copy()

skew_candidates = [c for c in ["capital_gain", "capital_loss", "hours_per_week"] if c in _train.columns]

for c in skew_candidates:
    raw_train = _train[c].clip(lower=0)
    _train[f"{c}_log1p"] = np.log1p(raw_train)
    _val[f"{c}_log1p"] = np.log1p(_val[c].clip(lower=0))
    _test[f"{c}_log1p"] = np.log1p(_test[c].clip(lower=0))

print("Added log-transformed columns:", [c for c in _train.columns if c.endswith("_log1p")])

# Before/after comparison
for c in skew_candidates[:2]:
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    _train[c].plot(kind="hist", bins=50, ax=axes[0], title=f"{c} (original)")
    _train[f"{c}_log1p"].plot(kind="hist", bins=50, ax=axes[1], title=f"{c}_log1p")
    plt.tight_layout()
    plt.show()

##### Power Transform

*What it does*: It learns a non-linear monotonic transformation, such as Yeo-Johnson, to make a feature more symmetric and stable.

*When to use it*: when a feature is strongly skewed and you want something more flexible than a fixed log transform.

*What to pay attention to*: it must be fit on train only and it is harder to interpret than the original scale. Yeo-Johnson can handle zeros and negatives, while Box-Cox only works on strictly positive inputs.

*Model implications*: it can improve linear and distance-based models on skewed data, but the extra complexity is often unnecessary for tree models.

📚 [Map data to a normal distribution](https://scikit-learn.org/stable/auto_examples/preprocessing/plot_map_data_to_normal.html)

In [ ]:
_train = X_train_num.copy()
_val = X_val_num.copy()
_test = X_test_num.copy()

if "capital_gain" in _train.columns:
    pt = PowerTransformer(method="yeo-johnson")
    _train["capital_gain_power"] = pt.fit_transform(_train[["capital_gain"]]).ravel()
    _val["capital_gain_power"] = pt.transform(_val[["capital_gain"]]).ravel()
    _test["capital_gain_power"] = pt.transform(_test[["capital_gain"]]).ravel()

print("Added power-transformed columns:", [c for c in _train.columns if c.endswith("_power")])

> **Do It Yourself**
>
> Pick one scaler for each numeric feature group and justify with distribution shape.
>
> Hint: use robust scaling when outliers are frequent; tree-based models often need little or no scaling.

###### Practical Scaling Recommendation

*What it does*: It builds a quick recommendation table based on skewness and outlier rate.

*When to use it*: when you want a simple first-pass policy before hand-tuning each feature group.

*What to pay attention to*: this is a heuristic, not a proof. Final scaling choices should still depend on model family and feature semantics.

*Model implications*: heuristics are useful for notebook exploration, but production pipelines should document the actual reason each scaler was chosen.


In [ ]:
scale_cols = X_train_num.columns.tolist()

scale_recommendations = []
for c in scale_cols[:20]:
    s = X_train_num[c]
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    outlier_rate = (((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).mean()) if iqr > 0 else 0
    skew = s.skew() if s.nunique() > 2 else 0

    if outlier_rate > 0.02:
        rec = "RobustScaler"
    elif abs(skew) > 1.0:
        rec = "log/Power + StandardScaler"
    else:
        rec = "StandardScaler"

    scale_recommendations.append({"column": c, "outlier_rate": round(float(outlier_rate), 4), "skew": round(float(skew), 3), "recommended": rec})

show(pd.DataFrame(scale_recommendations), n=20)

## <a id="engineering"></a> Section 7 - Feature Engineering (Advanced)

Include interactions, ratios, group aggregates, discretization, log transforms,
datetime features, polynomial expansion, and hashing trick.

📚 [PolynomialFeatures](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html) · [KBinsDiscretizer](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.KBinsDiscretizer.html) · [FeatureHasher](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.FeatureHasher.html) · [Feature-engine RelativeFeatures](https://feature-engine.trainindata.com/en/latest/user_guide/creation/RelativeFeatures.html) · [Feature-engine DatetimeFeatures](https://feature-engine.trainindata.com/en/latest/user_guide/datetime/DatetimeFeatures.html) · Weinberger et al. (2009), *Feature Hashing for Large Scale Multitask Learning, ICML*

##### Interaction and Ratio Features

These features encode the idea that the effect of one variable may depend on another, or that relative scale matters more than raw magnitude.

*What to pay attention to*: ratio features can explode when the denominator gets close to zero, so guard with offsets, clipping, or explicit missing/zero handling.

These usually help most when the downstream model is too simple to learn the relationship on its own, especially for linear models.

📚 Practitioner example: [Kaggle House Prices feature engineering tutorial](https://www.kaggle.com/code/ryanholbrook/feature-engineering-for-house-prices#Create-Features-with-Pandas) shows interaction-style features as part of a stronger competition pipeline.
📚 Deeper read: [AutoCross](https://arxiv.org/abs/1904.12857) explains why feature crosses can materially improve tabular models.

In [ ]:
# 1) Interaction + ratio features
_train = X_train_num.copy()
_val = X_val_num.copy()
_test = X_test_num.copy()

if {"capital_gain", "capital_loss"}.issubset(_train.columns):
    for _df in [_train, _val, _test]:
        _df["net_capital"] = _df["capital_gain"] - _df["capital_loss"]

if {"education_num", "age"}.issubset(_train.columns):
    for _df in [_train, _val, _test]:
        _df["education_per_age"] = _df["education_num"] / (_df["age"].abs() + 1)

new_cols = [c for c in _train.columns if c not in X_train_num.columns]
print("Interaction features added:", new_cols)
show(_train[new_cols].head())

##### Group Aggregates

Group aggregates add context, for example the average behavior of an occupation, store, customer segment, or zipcode.

*What to pay attention to*: learn the aggregates on train only, decide how to fill unseen groups, and be especially careful when the aggregate uses the target because that can leak very easily.

Check this [Kaggle notebook](https://www.kaggle.com/code/ryanholbrook/creating-features#Group-Transforms) for further material on features based on groups.

In [ ]:
# 2) Group aggregates learned on train
if {"occupation", "hours_per_week"}.issubset(train_df.columns):
    grp_hours = (train_df.assign(hours_num=to_numeric_loose(train_df["hours_per_week"]))
                 .groupby(train_df["occupation"].astype(str))["hours_num"].mean())
    grp_target = train_df.groupby(train_df["occupation"].astype(str))[TARGET_BIN_COL].mean()

    for raw_df, name in [(train_df, "train"), (val_df, "val"), (test_df, "test")]:
        key = raw_df["occupation"].astype(str)
        result = pd.DataFrame({
            "occupation_mean_hours": key.map(grp_hours).fillna(grp_hours.mean()),
            "occupation_target_rate_train": key.map(grp_target).fillna(grp_target.mean()),
        }, index=raw_df.index)
        print(f"{name} group aggregate features:")
        show(result.head(3))

##### Binning / Discretization

Binning can help when the relationship between a numeric feature and the target is non-linear, but the downstream model is simple.

*What to pay attention to*: min/max-based bins can be unstable under outliers, and too many bins can turn one feature into a wide sparse representation.

📚 Clean demo: [scikit-learn feature discretization example](https://scikit-learn.org/stable/auto_examples/preprocessing/plot_discretization_classification.html) shows how discretization can let a linear model recover a non-linear boundary.
📚 Deeper read: [On the Effectiveness of Discretizing Quantitative Attributes in Linear Classifiers](https://arxiv.org/abs/1701.07114) gives the rationale for why this can work.

In [ ]:
# 3) Binning / discretization
if "age" in X_train_num.columns:
    # Keep this demo runnable even if the missing-value section above was skipped.
    age_train = pd.DataFrame({"age": to_numeric_loose(X_train_num["age"])})
    age_val = pd.DataFrame({"age": to_numeric_loose(X_val_num["age"])})
    age_test = pd.DataFrame({"age": to_numeric_loose(X_test_num["age"])})
    age_fill = age_train["age"].median()
    for frame in [age_train, age_val, age_test]:
        frame["age"] = frame["age"].fillna(age_fill)

    kb = KBinsDiscretizer(n_bins=5, encode="ordinal", strategy="quantile")
    train_age_bin = kb.fit_transform(age_train.to_numpy(copy=True)).ravel()
    val_age_bin = kb.transform(age_val.to_numpy(copy=True)).ravel()
    test_age_bin = kb.transform(age_test.to_numpy(copy=True)).ravel()

    print("Age bin distribution (train):")
    show(pd.Series(train_age_bin, name="age_bin").value_counts().sort_index().to_frame())

##### Datetime-Derived Features

Raw timestamps are rarely the feature you want. Month, week, day of week, hour, weekend flags, or elapsed-time features are often more useful.

*What to pay attention to*: some calendar features are cyclical and some are pure process artifacts. For example, `hour=23` and `hour=0` are close in time even though their numeric values are far apart.

These features tend to help when behavior has daily, weekly, monthly, or seasonal structure and the raw timestamp hides that pattern.

📚 Practitioner example: [Bike Sharing Demand thesis/write-up](https://ke-tud.github.io/lehre/arbeiten/studien/2015/Dong_Ying.pdf) shows strong gains from extracting calendar components such as year, month, day of week, and hour.
📚 Notebook example: [Bike sharing feature engineering notebook](https://notebook.community/vsmolyakov/kaggle/bike_sharing/bike_notebook) uses derived time features that become among the most predictive variables.

In [ ]:
# 4) Datetime-derived features
if "record_written_at" in train_df.columns:
    for raw_df, name in [(train_df, "train"), (val_df, "val"), (test_df, "test")]:
        dt = pd.to_datetime(raw_df["record_written_at"], errors="coerce")
        result = pd.DataFrame({
            "record_month": dt.dt.month.fillna(0).astype(int),
            "record_dayofweek": dt.dt.dayofweek.fillna(0).astype(int),
            "record_hour": dt.dt.hour.fillna(0).astype(int),
        }, index=raw_df.index)
        if name == "train":
            print("Datetime-derived features (train):")
            show(result.head())

##### Polynomial Expansion

Polynomial features generate all pairwise interactions and squared terms from a set of numeric inputs. This lets linear models capture non-linear relationships without switching to a more complex algorithm.

*What to pay attention to*: the number of features grows quickly with degree and input count — `degree=2` on *k* features produces *k(k+1)/2* new columns. Higher degrees overfit fast on small datasets.

📚 Practitioner walkthrough: [How to Use Polynomial Feature Transforms for Machine Learning](https://machinelearningmastery.com/polynomial-features-transforms-for-machine-learning/) shows before/after scoring at degree 2, 3, and 4 and covers the overfitting trade-off.
📚 Kaggle notebook: [Data Science for Tabular Data — Advanced Techniques](https://www.kaggle.com/code/vbmokin/data-science-for-tabular-data-advanced-techniques) includes polynomial/interaction features as part of a realistic end-to-end pipeline.

In [ ]:
# 5) Polynomial expansion (compact subset)
poly_base = [c for c in ["age", "hours_per_week", "education_num"] if c in X_train_num.columns]
if len(poly_base) >= 2:
    # Keep this demo independent from earlier imputation cells.
    poly_train = X_train_num[poly_base].apply(to_numeric_loose)
    poly_val = X_val_num[poly_base].apply(to_numeric_loose)
    poly_test = X_test_num[poly_base].apply(to_numeric_loose)
    poly_fill = poly_train.median(numeric_only=True)
    poly_train = poly_train.fillna(poly_fill)
    poly_val = poly_val.fillna(poly_fill)
    poly_test = poly_test.fillna(poly_fill)

    poly = PolynomialFeatures(degree=2, include_bias=False)
    tr_poly = poly.fit_transform(poly_train.to_numpy(copy=True))
    va_poly = poly.transform(poly_val.to_numpy(copy=True))
    te_poly = poly.transform(poly_test.to_numpy(copy=True))

    poly_cols = [f"poly_{c}" for c in poly.get_feature_names_out(poly_base)]
    # Keep only interaction and squared terms (skip raw duplicates)
    keep_poly = [c for c in poly_cols if " " in c.replace("poly_", "") or "^2" in c]

    print(f"Polynomial features ({len(keep_poly)} interaction/squared terms):", keep_poly)
    show(pd.DataFrame(tr_poly, columns=poly_cols)[keep_poly].head())

##### Feature Hashing

Feature hashing (the "hashing trick") maps high-cardinality categoricals into a fixed-size numeric vector without building an explicit vocabulary. It trades a small collision risk for constant memory and the ability to handle unseen categories at inference time.

*What to pay attention to*: collisions are silent — two distinct categories can land in the same bucket. A practical rule of thumb is to set the hash space to roughly 20× the number of distinct categories to keep collision rates low.

📚 Practitioner walkthrough: [Feature Hashing for High Cardinality](https://medium.com/flutter-community/dealing-with-categorical-features-with-high-cardinality-feature-hashing-7c406ff867cb) covers `FeatureHasher` usage, collision trade-offs, and how to pick the number of bins.
📚 Comparison article: [4 Ways to Encode High-Cardinality Categoricals](https://towardsdatascience.com/4-ways-to-encode-categorical-features-with-high-cardinality-1bc6d8fd7b13/) benchmarks hashing against target encoding and binary encoding on the Criteo Kaggle dataset.

In [ ]:
# 6) Hashing trick for high-cardinality categorical
if "native_country" in train_df.columns:
    hasher = FeatureHasher(n_features=8, input_type="dict")

    def hash_series(series: pd.Series, prefix: str) -> pd.DataFrame:
        tokens = series.fillna("MISSING").astype(str).tolist()
        records = [{tok: 1.0} for tok in tokens]
        mat = hasher.transform(records)
        cols = [f"{prefix}_{i}" for i in range(mat.shape[1])]
        return pd.DataFrame(mat.toarray(), columns=cols, index=series.index)

    train_hash = hash_series(train_df["native_country"], "hash_native_country")
    val_hash = hash_series(val_df["native_country"], "hash_native_country")
    test_hash = hash_series(test_df["native_country"], "hash_native_country")

    print("Hashed features shape:", train_hash.shape)
    show(pd.concat([train_df[["native_country"]], train_hash], axis=1).head())

> **Do It Yourself**
>
> Propose 3 engineered features and write one hypothesis per feature.
>
> Hint: each feature should map to a behavioral or business explanation.

In [ ]:
feature_hypotheses = pd.DataFrame([
    {"feature": "net_capital", "hypothesis": "Net capital position (gain minus loss) captures overall wealth trajectory", "leakage_risk": "low"},
    {"feature": "education_per_age", "hypothesis": "Education intensity — same education level means different things at 25 vs 60", "leakage_risk": "low"},
    {"feature": "occupation_mean_hours", "hypothesis": "Occupation-level workload baseline adds contextual prior", "leakage_risk": "low (train-only aggregate)"},
    {"feature": "record_dayofweek", "hypothesis": "Application workflow timing may correlate with process outcomes", "leakage_risk": "medium (process artifact)"},
])
show(feature_hypotheses, n=10)

if "train_fe_df" in globals():
    num_arr = train_fe_df.select_dtypes(include=[np.number]).to_numpy(dtype=float)
    assert np.isfinite(num_arr).all()
    assert train_fe_df.isna().sum().sum() == 0
    print("Feature engineering sanity checks passed.")
else:
    print("Sanity checks run after the manual assembly cell below creates train_fe_df.")

##### A Minimal sklearn Pipeline

When you want a preprocessing recipe you can reuse safely, wrap it in a sklearn `Pipeline`.

- Fit it on **train only**.
- Reuse the same fitted object on **val/test**.
- Keep preprocessing logic in one place instead of manually repeating steps.
- In Day 2, you will attach a model as the last step and tune the whole object leakage-safely.

Below is a tiny example with numeric and categorical columns.

In [ ]:
# Minimal preprocessing Pipeline example
demo_num_cols = [c for c in ["age", "education_num", "hours_per_week"] if c in train_df.columns]
demo_cat_cols = [c for c in ["workclass", "occupation"] if c in train_df.columns]
demo_cols = demo_num_cols + demo_cat_cols

X_train_pipe_demo = train_df[demo_cols].copy()
X_val_pipe_demo = val_df[demo_cols].copy()
X_test_pipe_demo = test_df[demo_cols].copy()

for col in demo_num_cols:
    X_train_pipe_demo[col] = to_numeric_loose(X_train_pipe_demo[col])
    X_val_pipe_demo[col] = to_numeric_loose(X_val_pipe_demo[col])
    X_test_pipe_demo[col] = to_numeric_loose(X_test_pipe_demo[col])

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler()),
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocess_demo = ColumnTransformer([
    ("num", num_pipe, demo_num_cols),
    ("cat", cat_pipe, demo_cat_cols),
])

pipe_demo = Pipeline([
    ("preprocess", preprocess_demo),
])

Xt_demo = pipe_demo.fit_transform(X_train_pipe_demo)
Xv_demo = pipe_demo.transform(X_val_pipe_demo)
Xte_demo = pipe_demo.transform(X_test_pipe_demo)

demo_feature_names = pipe_demo.named_steps["preprocess"].get_feature_names_out()
train_pipe_demo_df = pd.DataFrame(Xt_demo, columns=demo_feature_names, index=X_train_pipe_demo.index)

show(train_pipe_demo_df, n=5)
print("Demo columns used:", demo_cols)
print("Train / val / test shapes:", Xt_demo.shape, Xv_demo.shape, Xte_demo.shape)
print("Key rule: fit on train only, then reuse the fitted pipeline on val/test.")


##### Manual Pipeline Assembly

Now we combine the techniques into a single processed dataset for feature selection.
This cell is notebook-style chaining, not a sklearn `Pipeline` object.
It is useful for showing the logic step by step, but the small example above is the reusable production pattern.

In [ ]:
# Assemble processed data for feature selection (Section 8+)
# 1) Start from numeric base
_train = X_train_num.apply(to_numeric_loose).copy()
_val = X_val_num.apply(to_numeric_loose).copy()
_test = X_test_num.apply(to_numeric_loose).copy()

# Keep this assembly cell self-contained even if earlier missing-value cells were skipped.
for df in [_train, _val, _test]:
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

_base_medians = _train.median(numeric_only=True)
for col in _train.columns:
    med = _base_medians.get(col, 0.0)
    _train[col] = _train[col].fillna(med)
    _val[col] = _val[col].fillna(med)
    _test[col] = _test[col].fillna(med)

# 2) Robust scaling
_rb = RobustScaler()
_scale_cols = _train.columns.tolist()
_train[_scale_cols] = _rb.fit_transform(_train[_scale_cols])
_val[_scale_cols] = _rb.transform(_val[_scale_cols])
_test[_scale_cols] = _rb.transform(_test[_scale_cols])

# 3) Interaction features
if {"capital_gain", "capital_loss"}.issubset(_train.columns):
    for _df in [_train, _val, _test]:
        _df["net_capital"] = _df["capital_gain"] - _df["capital_loss"]
if {"education_num", "age"}.issubset(_train.columns):
    for _df in [_train, _val, _test]:
        _df["education_per_age"] = _df["education_num"] / (_df["age"].abs() + 1)

# 4) Group aggregates
if {"occupation", "hours_per_week"}.issubset(train_df.columns):
    grp_hours = (train_df.assign(hours_num=to_numeric_loose(train_df["hours_per_week"]))
                 .groupby(train_df["occupation"].astype(str))["hours_num"].mean())
    grp_target = train_df.groupby(train_df["occupation"].astype(str))[TARGET_BIN_COL].mean()
    for raw, out in [(train_df, _train), (val_df, _val), (test_df, _test)]:
        key = raw["occupation"].astype(str)
        out["occupation_mean_hours"] = key.map(grp_hours).fillna(grp_hours.mean())
        out["occupation_target_rate_train"] = key.map(grp_target).fillna(grp_target.mean())

# 5) Binning
if "age" in _train.columns:
    kb = KBinsDiscretizer(n_bins=5, encode="ordinal", strategy="quantile")
    _train["age_bin"] = kb.fit_transform(_train[["age"]].to_numpy(copy=True)).ravel()
    _val["age_bin"] = kb.transform(_val[["age"]].to_numpy(copy=True)).ravel()
    _test["age_bin"] = kb.transform(_test[["age"]].to_numpy(copy=True)).ravel()

# 6) Datetime features
if "record_written_at" in train_df.columns:
    for raw, out in [(train_df, _train), (val_df, _val), (test_df, _test)]:
        dt = pd.to_datetime(raw["record_written_at"], errors="coerce")
        out["record_month"] = dt.dt.month.fillna(0).astype(int)
        out["record_dayofweek"] = dt.dt.dayofweek.fillna(0).astype(int)
        out["record_hour"] = dt.dt.hour.fillna(0).astype(int)

# 7) Polynomial (interaction + squared terms only)
poly_base = [c for c in ["age", "hours_per_week", "education_num"] if c in _train.columns]
if len(poly_base) >= 2:
    poly = PolynomialFeatures(degree=2, include_bias=False)
    tr_poly = poly.fit_transform(_train[poly_base].to_numpy(copy=True))
    va_poly = poly.transform(_val[poly_base].to_numpy(copy=True))
    te_poly = poly.transform(_test[poly_base].to_numpy(copy=True))
    poly_cols = [f"poly_{c}" for c in poly.get_feature_names_out(poly_base)]
    keep_poly = [c for c in poly_cols if " " in c.replace("poly_", "") or "^2" in c]
    for src, out in [(tr_poly, _train), (va_poly, _val), (te_poly, _test)]:
        tmp = pd.DataFrame(src, columns=poly_cols, index=out.index)
        for pc in keep_poly:
            out[pc] = tmp[pc]

# 8) Feature hashing
if "native_country" in train_df.columns:
    hasher = FeatureHasher(n_features=8, input_type="dict")
    def hash_series(series, prefix):
        tokens = series.fillna("MISSING").astype(str).tolist()
        records = [{tok: 1.0} for tok in tokens]
        mat = hasher.transform(records)
        cols = [f"{prefix}_{i}" for i in range(mat.shape[1])]
        return pd.DataFrame(mat.toarray(), columns=cols, index=series.index)
    for raw, out in [(train_df, _train), (val_df, _val), (test_df, _test)]:
        hashed = hash_series(raw["native_country"], "hash_native_country")
        for hc in hashed.columns:
            out[hc] = hashed[hc].values

# 9) Final cleanup
for df in [_train, _val, _test]:
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

_medians = _train.median(numeric_only=True)
for col in _train.columns:
    med = _medians.get(col, 0.0)
    _train[col] = _train[col].fillna(med)
    _val[col] = _val[col].fillna(med)
    _test[col] = _test[col].fillna(med)

train_fe_df = _train
val_fe_df = _val
test_fe_df = _test

print("Assembled feature shapes:", train_fe_df.shape, val_fe_df.shape, test_fe_df.shape)

num_arr = train_fe_df.select_dtypes(include=[np.number]).to_numpy(dtype=float)
assert np.isfinite(num_arr).all()
assert train_fe_df.isna().sum().sum() == 0
print("Feature engineering sanity checks passed.")

## <a id="selection"></a> Section 8 - Feature Selection / Dimensionality Reduction (Qualitative)

No model-score benchmark here. We use qualitative selectors and interpretive outputs to decide which features to keep and which to drop.

The techniques below are ordered from simplest (unsupervised, zero parameters) to more complex (model-based, supervised). In practice you would combine several of them — no single method catches everything.

| Technique | Supervised? | What it catches |
|---|---|---|
| **Variance Threshold** | No | Constant or near-constant columns |
| **Correlation Pruning** | No | Redundant feature pairs |
| **Mutual Information** | Yes | Non-linear association with the target |
| **Model-based Importance** | Yes | Predictive contribution inside an ensemble |
| **PCA** | No | Directions of maximum variance (linear) |

**Other techniques worth knowing** (not demonstrated here, but commonly used in practice):

- **[t-SNE](https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html)** — non-linear embedding for 2D/3D visualisation of cluster structure. Good for EDA, but not for feature selection (non-invertible, no `.transform` for new data).
- **[UMAP](https://umap-learn.readthedocs.io/)** — similar goal to t-SNE but faster, better at preserving global structure, and supports transforming unseen points. Increasingly the default choice for visualising high-dimensional data.
- **[Predictive Power Score (PPS)](https://github.com/8080labs/ppscore)** — an asymmetric, model-agnostic score that measures how well feature X predicts target Y. Unlike correlation, it captures non-linear relationships and works with categoricals. Think of it as "mutual information made easy to interpret."

📚 [scikit-learn Feature Selection Guide](https://scikit-learn.org/stable/modules/feature_selection.html) · [VarianceThreshold](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.VarianceThreshold.html) · [mutual_info_classif](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.mutual_info_classif.html) · [PCA](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)

In [ ]:
# Use numeric matrix for selectors.
X_sel_train = train_fe_df.select_dtypes(include=[np.number]).copy()
X_sel_val = val_fe_df[X_sel_train.columns].copy()
X_sel_test = test_fe_df[X_sel_train.columns].copy()

print(f"Starting feature matrix: {X_sel_train.shape[1]} columns")

##### 1) Variance Threshold

The simplest filter: drop any feature whose variance is zero (or below a chosen threshold). A constant column carries no information for any model. This is a cheap first pass that removes obvious dead weight before more expensive methods run.

In [ ]:
# 1) Variance threshold — remove constant / near-constant features
vt = VarianceThreshold(threshold=0.0)
vt.fit(X_sel_train)
vt_cols = X_sel_train.columns[vt.get_support()].tolist()
dropped_vt = [c for c in X_sel_train.columns if c not in vt_cols]

print(f"Kept {len(vt_cols)} / {X_sel_train.shape[1]} features (dropped {len(dropped_vt)} zero-variance)")
if dropped_vt:
    print("Dropped:", dropped_vt)

##### 2) Correlation Pruning

When two features are nearly perfectly correlated (|r| > 0.95), they carry almost the same information. Keeping both wastes capacity and can destabilise linear models (multicollinearity). We scan the upper triangle of the correlation matrix and drop one from each highly-correlated pair.

In [ ]:
# 2) Correlation pruning — drop one from each highly-correlated pair
corr = X_sel_train[vt_cols].corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
remove_corr = [col for col in upper.columns if (upper[col] > 0.95).any()]
corr_pruned_cols = [c for c in vt_cols if c not in remove_corr]

print(f"Kept {len(corr_pruned_cols)} / {len(vt_cols)} features (dropped {len(remove_corr)} highly-correlated)")
if remove_corr:
    print("Dropped:", remove_corr)

##### 3) Mutual Information

Mutual information (MI) measures how much knowing feature X reduces uncertainty about target Y. Unlike Pearson correlation it captures **non-linear** relationships and works for both continuous and categorical targets. Features with MI close to zero are essentially noise with respect to the target.

In [ ]:
# 3) Mutual information ranking
mi = mutual_info_classif(X_sel_train[corr_pruned_cols], y_train, random_state=SEED)
mi_rank = pd.DataFrame({"feature": corr_pruned_cols, "mi": mi}).sort_values("mi", ascending=False)

print(f"Mutual information scores for {len(corr_pruned_cols)} features:")
show(mi_rank, n=15)

##### 4) Model-Based Importance

Train a quick Random Forest and read off `feature_importances_`. This tells you which features the ensemble actually *used* to split on. It is supervised and captures interactions the forest discovers internally, but it is biased towards high-cardinality and continuous features — so treat it as one signal among several, not ground truth.

In [ ]:
# 4) Model-based importance (illustrative only)
rf = RandomForestClassifier(n_estimators=250, random_state=SEED, n_jobs=-1)
rf.fit(X_sel_train[corr_pruned_cols], y_train)
imp_rank = pd.DataFrame({
    "feature": corr_pruned_cols,
    "importance": rf.feature_importances_,
}).sort_values("importance", ascending=False)

print(f"Random Forest feature importances ({len(corr_pruned_cols)} features):")
show(imp_rank, n=15)

##### 5) PCA Projection

PCA finds the directions of maximum variance in the feature matrix and projects the data onto them. It is **unsupervised** — it does not look at the target — so the top principal components explain the most *spread* in the data, which may or may not align with what is predictive. The scree plot (cumulative variance) tells you how many components you need to retain most of the information, and the scatter plot gives a quick visual check for class separability.

In [ ]:
# 5) PCA projection
pca_input_cols = corr_pruned_cols[: min(80, len(corr_pruned_cols))]
std_for_pca = StandardScaler()
X_pca_train = std_for_pca.fit_transform(X_sel_train[pca_input_cols])

pca = PCA(n_components=min(10, X_pca_train.shape[1]), random_state=SEED)
pcs = pca.fit_transform(X_pca_train)

pca_var = pd.DataFrame({
    "component": np.arange(1, len(pca.explained_variance_ratio_) + 1),
    "explained_variance_ratio": pca.explained_variance_ratio_,
    "cumulative": np.cumsum(pca.explained_variance_ratio_),
})
show(pca_var, n=10)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].plot(pca_var["component"], pca_var["cumulative"],
marker="o")
axes[0].set_title("PCA cumulative explained variance")
axes[0].set_xlabel("component")
axes[0].set_ylabel("cumulative variance")

scatter_idx = np.random.RandomState(SEED).choice(len(pcs),
size=min(2000, len(pcs)), replace=False)
for label in sorted(y_train.unique()):
    mask = y_train.iloc[scatter_idx] == label
    axes[1].scatter(pcs[scatter_idx[mask], 0],
pcs[scatter_idx[mask], 1],
                    alpha=0.3, s=10, label=label)
axes[1].set_title("PCA projection (PC1 vs PC2)")
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")
axes[1].legend(title="Target")
plt.tight_layout()
plt.show()

selected_features_v1 = corr_pruned_cols

> **Do It Yourself**
>
> Choose a candidate subset by combining at least two methods (e.g., MI + correlation pruning).
>
> Hint: keep interpretability in mind, not only ranking position.

In [ ]:
top_mi = mi_rank.head(20)["feature"].tolist()
top_imp = imp_rank.head(20)["feature"].tolist()

candidate_subset = sorted(set(top_mi).intersection(set(top_imp)))
if len(candidate_subset) < 10:
    candidate_subset = sorted(set(top_mi[:10] + top_imp[:10]))

print("Candidate subset size:", len(candidate_subset))
print("Candidate subset sample:", candidate_subset[:20])


## <a id="ames-pipeline"></a> Section 9 - End-to-End Pipeline (Ames)

This section puts everything from Sections 3–8 together into a single sklearn `Pipeline` + `ColumnTransformer` on a **different dataset** (Ames Housing, regression).

The goal is to practice *transferring* the same reasoning — impute, encode, scale, engineer — to a new problem with different column types, and to package it in the industry-standard way: a fitted pipeline object you can `.transform()` on unseen data without any manual steps.

> **Do It Yourself**
>
> Before looking at the reference solution:
> 1. Load the Ames clean dataset and inspect its columns.
> 2. Decide which columns are numeric, which are categorical, and which to drop.
> 3. Sketch (on paper or in a cell) which preprocessing steps each group needs.
> 4. Then compare your plan with the pipeline below.

In [ ]:
# Load Ames clean dataset
ames = pd.read_csv("day1/generated/ames_housing_clean.csv")
print("Ames shape:", ames.shape)
show(ames, n=5)
print("\nColumn dtypes:")
print(ames.dtypes.value_counts())

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer

# --- Column selection ---
drop_cols = ["property_id", "id"]
target_col = "sale_price"

y_ames = ames[target_col].copy()
X_ames = ames.drop(columns=drop_cols + [target_col], errors="ignore")

# Identify column types
cat_cols = X_ames.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X_ames.select_dtypes(include=["number"]).columns.tolist()

print(f"Numeric features: {len(num_cols)}")
print(f"Categorical features: {len(cat_cols)}")
print(f"Target: {target_col}")

# --- Train / val split (simple random for demo) ---
from sklearn.model_selection import train_test_split

X_tr, X_va, y_tr, y_va = train_test_split(
    X_ames, y_ames, test_size=0.2, random_state=SEED
)
print(f"\nTrain: {X_tr.shape}, Val: {X_va.shape}")

##### Building the Pipeline

Each branch of the `ColumnTransformer` handles one column type with the same techniques we used in Sections 4–6:

- **Numeric**: impute with median → scale with `StandardScaler`
- **Categorical**: impute with a constant `"MISSING"` → one-hot encode (dropping the first level to avoid multicollinearity, and handling unknown categories at transform time)

In [ ]:
# --- Numeric branch: impute → scale ---
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

# --- Categorical branch: impute → one-hot ---
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="MISSING")),
    ("onehot", OneHotEncoder(drop="first", handle_unknown="infrequent_if_exist",
                             sparse_output=False, min_frequency=5)),
])

# --- Combine ---
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, num_cols),
    ("cat", categorical_pipeline, cat_cols),
], remainder="drop")

preprocessor

##### Fit and Inspect

We fit the pipeline on the training set only, then transform both splits. The output is a clean numeric matrix ready for any model.

In [ ]:
# Fit on train, transform both splits
X_tr_processed = preprocessor.fit_transform(X_tr)
X_va_processed = preprocessor.transform(X_va)

feature_names = preprocessor.get_feature_names_out()
X_tr_df = pd.DataFrame(X_tr_processed, columns=feature_names, index=X_tr.index)
X_va_df = pd.DataFrame(X_va_processed, columns=feature_names, index=X_va.index)

print(f"Processed train shape: {X_tr_df.shape}")
print(f"Processed val shape:   {X_va_df.shape}")
print(f"Any NaN in train: {X_tr_df.isna().any().any()}")
print(f"Any NaN in val:   {X_va_df.isna().any().any()}")
print(f"\nFeature name samples: {feature_names[:5].tolist()} ... {feature_names[-5:].tolist()}")
show(X_tr_df, n=5)

##### Quick Sanity Check

A fast linear model confirms the pipeline produces usable features. This is **not** a modeling exercise — just a smoke test that the preprocessing didn't break anything obvious.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score

ridge = Ridge(alpha=1.0)
ridge.fit(X_tr_df, y_tr)

y_pred_tr = ridge.predict(X_tr_df)
y_pred_va = ridge.predict(X_va_df)

print("Ridge sanity check (no tuning, just verifying pipeline output):")
print(f"  Train  R² = {r2_score(y_tr, y_pred_tr):.3f},  MAE = {mean_absolute_error(y_tr, y_pred_tr):,.0f}")
print(f"  Val    R² = {r2_score(y_va, y_pred_va):.3f},  MAE = {mean_absolute_error(y_va, y_pred_va):,.0f}")

### Model-vs-Preprocessing Summary

Use this as a quick translation layer between Day 1 preprocessing choices and the model families you will compare in Day 2.

| Choice | Logistic Regression | kNN / SVM / MLP | Decision Tree | Random Forest | HistGB / XGBoost / LightGBM | CatBoost |
|---|---|---|---|---|---|---|
| Missing values | Must handle first | Must handle first | Must handle first in sklearn tree | Must handle first in sklearn RF | Native handling often available | Native handling available |
| Raw categorical strings | Must encode | Must encode | Must encode in sklearn | Must encode in sklearn | Usually must encode in sklearn/XGB/LGBM workflows | Native categorical handling possible |
| Scaling | Usually important | Very important | Usually not needed | Usually not needed | Usually not needed | Usually not needed |
| Sensitive to outliers | Yes | Yes, often strongly | Less sensitive | Less sensitive | Less sensitive | Less sensitive |
| Log / power transform | Often useful on skewed features | Often useful | Often optional | Often optional | Often optional | Often optional |
| One-hot encoding | Good for low-cardinality features | Good for low-cardinality features | Works, but can get wide | Works, but can get wide | Works, but high-cardinality needs care | Often not preferred |
| High-cardinality categoricals | Risk of wide sparse matrix / overfit | Same problem, often worse | Can memorize rare levels after OHE | Can still overfit noisy sparse levels | Often better with rare-level handling or supervised encodings | Often strongest here |
| Train-only fitted preprocessing | Required | Required | Required | Required | Required | Required |

| Feature engineering choice | Linear / distance models | Tree ensembles | What to remember |
|---|---|---|---|
| Interaction features | Often very helpful | Less necessary | Linear models cannot invent them easily |
| Ratio features | Often helpful | Can help | Guard against zero or near-zero denominators |
| Group aggregates | Can help a lot | Can help a lot | Learn them on train only; target-based versions can leak badly |
| Datetime-derived features | Helpful | Helpful | Month / week / day-of-week are often better than raw timestamps |
| Binning / discretization | Can help | Often unnecessary | Can improve simple models, but can also throw away signal |
| PCA / dimensionality reduction | Sometimes useful | Rarely needed | PCA preserves variance, not necessarily predictive signal |
| Feature hashing | Useful for very wide sparse inputs | Sometimes useful | Mostly for high-cardinality categoricals or text-like features |

- `Must handle first` means the model will fail or behave badly if you skip the step.
- `Usually not needed` does **not** mean `never useful`.
- Tree-based models are more forgiving, but they are not magic: leakage, impossible values, and bad split strategy still matter.


## <a id="checklist"></a> Section 10 - End-of-Notebook Preprocessing Checklist

> **Do It Yourself**
>
> Fill this checklist for one dataset before moving to modeling:

> - [ ] Split strategy is deployment-consistent (time/group aware if needed).
> - [ ] Column typing is validated (numeric/categorical/date/text/process).
> - [ ] Leakage columns are excluded from features.
> - [ ] Categorical encoding choices documented by feature type.
> - [ ] Missingness policy documented (with indicators where useful).
> - [ ] Invalid/outlier policy documented (rule/statistical/model-based).
> - [ ] Scaling/transform choices documented by model sensitivity.
> - [ ] Engineered features have explicit business hypotheses.
> - [ ] Feature selection logic combines at least 2 methods.
> - [ ] Deployment assumptions are explicitly written.

In [ ]:
student_checklist_template = {
    "dataset": "adult_income_issues.csv",
    "split_strategy": "use provided split + train/val inside train",
    "typing_done": False,
    "leakage_checked": False,
    "encoding_policy_done": False,
    "missingness_policy_done": False,
    "outlier_policy_done": False,
    "scaling_policy_done": False,
    "feature_hypotheses_done": False,
    "selection_policy_done": False,
    "deployment_assumptions_written": False,
}

student_checklist_template


In [ ]:
reference_checklist = {
    "dataset": "adult_income_issues.csv",
    "split_strategy": "Use provided split to preserve unseen test categories; derive val from train only.",
    "typing_done": True,
    "leakage_checked": True,
    "encoding_policy_done": True,
    "missingness_policy_done": True,
    "outlier_policy_done": True,
    "scaling_policy_done": True,
    "feature_hypotheses_done": True,
    "selection_policy_done": True,
    "deployment_assumptions_written": True,
}

pd.DataFrame(reference_checklist, index=[0])


---

## References

### Encoding
- scikit-learn, [Encoding Categorical Features](https://scikit-learn.org/stable/modules/preprocessing.html#encoding-categorical-features)
- category_encoders, [Library Documentation](https://contrib.scikit-learn.org/category_encoders/)

### Imputation
- scikit-learn, [Imputation of Missing Values](https://scikit-learn.org/stable/modules/impute.html)
- Rubin, D. B. (1976). Inference and Missing Data. *Biometrika*, 63(3), 581–592.
- van Buuren, S. & Groothuis-Oudshoorn, K. (2011). MICE: Multivariate Imputation by Chained Equations in R. *Journal of Statistical Software*, 45(3).

### Outlier Detection
- Liu, F. T., Ting, K. M., & Zhou, Z.-H. (2008). Isolation Forest. *ICDM*.

### Scaling & Transformations
- Yeo, I.-K. & Johnson, R. A. (2000). A New Family of Power Transformations to Improve Normality or Symmetry. *Biometrika*, 87(4), 954–959.

### Feature Engineering
- Weinberger, K., Dasgupta, A., Langford, J., Smola, A., & Attenberg, J. (2009). Feature Hashing for Large Scale Multitask Learning. *ICML*.
- Zheng, A. & Casari, A. (2018). *Feature Engineering for Machine Learning*. O'Reilly Media.

### Feature Selection
- scikit-learn, [Feature Selection Guide](https://scikit-learn.org/stable/modules/feature_selection.html)
- Guyon, I. & Elisseeff, A. (2003). An Introduction to Variable and Feature Selection. *JMLR*, 3, 1157–1182.

## <a id="acceptance"></a> Acceptance Checks
These checks validate the Day 1 preprocessing/feature-engineering workflow end-to-end.

In [ ]:
# 1) Split integrity
assert len(set(train_df[ID_COLS[0]]) & set(val_df[ID_COLS[0]])) == 0
assert len(set(train_df[ID_COLS[0]]) & set(test_df[ID_COLS[0]])) == 0

# 2) Leakage controls
for leak_col in LEAKAGE_COLS:
    assert leak_col not in train_fe_df.columns, f"Leakage col found in features: {leak_col}"

# 3) Base data unchanged (X_train_raw should not have extra columns from technique cells)
assert "hours_band" not in X_train_raw.columns, "X_train_raw was mutated by a technique cell"

# 4) Feature engineering sanity
num_arr = train_fe_df.select_dtypes(include=[np.number]).to_numpy(dtype=float)
assert np.isfinite(num_arr).all()

# 5) Feature selection outputs
assert len(candidate_subset) > 0
assert "explained_variance_ratio" in pca_var.columns

# 6) Reproducibility hooks
assert SEED == 42

print("All acceptance checks passed.")
print("Notebook is ready for Day 1 delivery.")